# Filtración de datos UNGRD

Autor: Andrés Felipe Sierra

Este código está diseñado para filtrar base de datos para luego extraelo en un csv, así ser reutilizable en un caso concreto.



## Librerias y funciones

**Librerias**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import xlrd
import openpyxl

**Funciones**

In [2]:
# Función para obtener los datos
def obtener_datos(url):
  data = pd.read_excel(url,skiprows = 4, sheet_name= "REPORTE DE EMERGENCIAS")
  return data

In [3]:
# Función para obtener los datos
def obtener_datos_para_modificados(url):
  data = pd.read_excel(url,skiprows = 3, sheet_name= "REPORTE DE EMERGENCIAS")
  return data

In [4]:
# Función alterna para obtener datos
def obtener_datos_alterno(url):
  data = pd.read_excel(url,skiprows=1,sheet_name= "REPORTE DE EMERGENCIAS")
  return data

In [5]:
# Función para ver la información del dataset
def info_dataset(da):
  da.info()

In [6]:
# Función para ver datos nutos
def datos_nulos(da):
  print(da.isnull().sum())

In [7]:
# Función para ver en porcentaje los nulos de un dataframe
def porcentaje_nulos(da):
  print(da.isnull().mean())

In [8]:
# Función para extraer las columnas necesarias de un dataset
def extraer_columnas(da,columnas):
  da = da[columnas]
  return da

In [9]:
# Función para borrar columnas
def borrar_columnas(da,columnas):
  da = da.drop(columnas, axis=1)
  return da

In [10]:
# Función para filtrar los datos
def filtrar_datos(da,columna,valor):
  da = da[da[columna] == valor]
  return da

In [11]:
# Función para extraer el año y mes de una variable de fecha
def extraer_año_mes(da, columna):
  da[columna + '_año'] = da[columna].dt.year
  da[columna + '_mes'] = da[columna].dt.month
  return da

In [12]:
def filtrar_por_palabras_clave(df):
  # Creamos un filtro por palabras clave (insensible a mayúsculas)
  pal_cla = ["cultivo", "cultivos", "agricultura"]
  # Como hay varias columnas llamadas "OTROS", solo se filtrará la que ya está disponible
  otros_cols = ['OTROS']
  comentarios_col = 'COMENTARIOS'

  # Crear un filtro por palabras clave (insensible a mayúsculas)
  keywords = pal_cla

  # Construir filtro sobre columnas OTROS y COMENTARIOS
  mask = pd.Series(False, index=df.index) # Use df.index instead of df_24.index
  for col in otros_cols + [comentarios_col]:
      mask |= df[col].astype(str).str.lower().str.contains("|".join(keywords), na=False) # Use df[col] instead of df_24[col]
  return df[mask]

In [13]:
# Función para unir las columnas con las columnas correspondientes
def unir_columnas(da,columna1,columna2):
  da[columna1 + columna2] = da[columna1].astype(str) + da[columna2].astype(str)
  return da

In [14]:
# Función para cambiar el nombre de una columna
def cambiar_nombre_columna(da,columna1,columna2):
  da = da.rename(columns={columna1: columna2})
  return da

In [15]:
# Función para unir datasets y reorganziar los indices
def unir_datasets(da1,da2):
  da = pd.concat([da1, da2])
  da = da.reset_index(drop=True)
  return da

In [16]:
# Función para descargar los datos en un archivo csv
def descargar_csv(da,nombre):
  da.to_csv(nombre, index=False)

In [17]:
# Función para descargar los datos en un archivos .xlsx
def descargar_xlsx(da,nombre):
  da.to_excel(nombre, index=False)

## Exploración de datos y preparación de datos

Cargamos los datasets para su exploración y luego la preparación

#### 2024

##### Exploración

In [18]:
data_2024 = obtener_datos("../Data/UNGRD/REPORTE-ANUAL-DE-EMERGENCIAS-2024.xls")

In [19]:
info_dataset(data_2024)

<class 'pandas.DataFrame'>
RangeIndex: 10730 entries, 0 to 10729
Data columns (total 79 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0                                        92 non-null     str           
 1   ..                                   0 non-null      float64       
 2   …                                    0 non-null      float64       
 3   FECHA                                10578 non-null  datetime64[us]
 4   DEPARTAMENTO                         10578 non-null  str           
 5   MUNICIPIO                            10578 non-null  str           
 6   EVENTO                               10578 non-null  str           
 7   CODIFICACIÓN SEGUN DIVIPOLA          10578 non-null  float64       
 8   COORDENADAS                          31 non-null     str           
 9   FAMILIAS                             11 non-null     float64       
 10  PERSONAS             

In [20]:
import math

cols = data_2024.columns.tolist()
total_cols = len(cols)
# Calculamos cuántas filas necesitamos (división hacia arriba)
filas = math.ceil(total_cols / 3) 

print(f"{'COLUMNA 1'.ljust(40)} | {'COLUMNA 2'.ljust(40)} | {'COLUMNA 3'}")
print("-" * 120)

for i in range(filas):
    col1 = cols[i] if i < total_cols else ""
    col2 = cols[i + filas] if (i + filas) < total_cols else ""
    col3 = cols[i + (filas * 2)] if (i + (filas * 2)) < total_cols else ""
    
    # Usamos ljust para que las tres columnas queden alineadas
    print(f"{col1.ljust(40)} | {col2.ljust(40)} | {col3}")

print("\n" + "-"*40)
print(f"Cantidad total de columnas: {total_cols}")

COLUMNA 1                                | COLUMNA 2                                | COLUMNA 3
------------------------------------------------------------------------------------------------------------------------
                                         | OTROS                                    | INVERSION 
..                                       | FECHA ACTIVACION                         | DESCRIPCION .1
…                                        | KITS DE ALIMENTO                         | TIPO 
FECHA                                    | KITS DE ASEO                             | DIAS
DEPARTAMENTO                             | KITS DE COCINA                           | VALOR .3
MUNICIPIO                                |  COLCHONETAS                             | OBJETO DE LA OBRA
EVENTO                                   | FRAZADAS                                 | VALOR .4
CODIFICACIÓN SEGUN DIVIPOLA              | CARPAS TIPO CAMPING                      | ACCIONES ESPECIFICAS
C

Hay un total de 78 columnas con 10729 registros, por otro lado, se observa que sus columnas poseen gran cantidad de datos nulos lo que hace preocupante la dinamica de la UNGRD (Unidad Nacional para la Gestión del Riesgo de Desastres) para recolectar datos.

Bajo el análisis del dataset desde Excel y en Python determinamos que no es necesario mantener algunas columnas cuando son irrelevantes, además esta sección de datasets servirán para determinar dónde usualmente ocurre derrumbes e inundaciones; Por otro lado, podemos ver en los comentarios que estás estan reportadas por centros poblados no por el main principal del proyecto, sin embargo, es útil para mapear dónde ocurren.

In [21]:
# Columnas seleccionadas
nom_col = ["FECHA", "CODIFICACIÓN SEGUN DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS.1",
           "FAMILIAS.1", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1", "PERSONAS", "FAMILIAS"]
# Se extraen
df_24 = extraer_columnas(data_2024, nom_col)

In [22]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_24)

<class 'pandas.DataFrame'>
RangeIndex: 10730 entries, 0 to 10729
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        10578 non-null  datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  10578 non-null  float64       
 2   DEPARTAMENTO                 10578 non-null  str           
 3   MUNICIPIO                    10578 non-null  str           
 4   EVENTO                       10578 non-null  str           
 5   CODIGO EMERGENCIA            10578 non-null  float64       
 6   PERSONAS.1                   1983 non-null   float64       
 7   FAMILIAS.1                   1714 non-null   float64       
 8   HECTAREAS                    6284 non-null   float64       
 9   VIV.DESTRU.                  248 non-null    object        
 10  VIV.AVER.                    1434 non-null   float64       
 11  VIAS                         599 non-null    float64

##### Preparación

Sabiendo que se redujo de 79 columnas a 13, esto hace más facil su preparación, además tener en cuenta que está sujeto a cambios y no todos los datasets se comportan igual. Agregando que estas columnas fueron seleccionadas con la idea se ser importante al momento se hacer una exploración más exahustiva.

¿Qué se va a hacer?
Se va a extraer el año y el mes para determinar si hay un comportamiento recurrente en cuanto a las fechas de la afectación, así podremos hacer una simulación del comportamiento a pesar del cambio climatico.

In [23]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_24 = extraer_año_mes(df_24,"FECHA")

In [24]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_24.copy()
df_24 = filtrar_datos(df_24,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [25]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_24)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 640 entries, 63 to 10575
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        640 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  640 non-null    float64       
 2   DEPARTAMENTO                 640 non-null    str           
 3   MUNICIPIO                    640 non-null    str           
 4   EVENTO                       640 non-null    str           
 5   CODIGO EMERGENCIA            640 non-null    float64       
 6   PERSONAS.1                   500 non-null    float64       
 7   FAMILIAS.1                   503 non-null    float64       
 8   HECTAREAS                    46 non-null     float64       
 9   VIV.DESTRU.                  25 non-null     object        
 10  VIV.AVER.                    422 non-null    float64       
 11  VIAS                         76

Este dataset no posee datos de deslizamientos, además otro punto para tener en cuenta en Colombia usualmente se registran o reportan más deslizamientos en las vías, por otro lado, por las ya las problematicas expuestas en los seguros de agrucultura estos no se registran o simplemente pasan desapercivido a pesar del aviso.

In [26]:
# Borramos la fehca
# df_24 = borrar_columnas(df_24,"FECHA") Lo dejamos por si acaso

In [27]:
info_dataset(df_24)

<class 'pandas.DataFrame'>
Index: 640 entries, 63 to 10575
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        640 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  640 non-null    float64       
 2   DEPARTAMENTO                 640 non-null    str           
 3   MUNICIPIO                    640 non-null    str           
 4   EVENTO                       640 non-null    str           
 5   CODIGO EMERGENCIA            640 non-null    float64       
 6   PERSONAS.1                   500 non-null    float64       
 7   FAMILIAS.1                   503 non-null    float64       
 8   HECTAREAS                    46 non-null     float64       
 9   VIV.DESTRU.                  25 non-null     object        
 10  VIV.AVER.                    422 non-null    float64       
 11  VIAS                         76 non-null     float64      

Quedaría un nuevo dataframe de 14 columnas con 640 observaciones, de las cuales 5 de ellas son objetos, entiendase como string, como DEPARTAMENTO, MUNICIPIO, EVENTO, OTROS y COMENTARIOS, estos nos ayudarán a indeficar dentro de inundaciones si está afectaciones a cultivos.

Bajo cómo quedó y la información que ofrece el dataset más reciente, se decide trabajar con las mismas columnas, a menos que salga alguna novedad que pueda a ayudar a un mejor contenido.

##### Segundo análisis de los datos

Como se exploró anteriormente todos los dataset para entender como se organizaban los dataframes se determina eliminar en los primeros dataset las columnas de FAMILIAS y personas que en la sección de RUD poseen la información diferente, ergo, no hace legible el contenido.

In [28]:
CO = ["PERSONAS","FAMILIAS"]
df_24 = borrar_columnas(df_24,CO)

In [29]:
print("FILAS/COLUMNAS",df_24.shape)
datos_nulos(df_24)

FILAS/COLUMNAS (640, 18)
FECHA                            0
CODIFICACIÓN SEGUN DIVIPOLA      0
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                0
PERSONAS.1                     140
FAMILIAS.1                     137
HECTAREAS                      594
VIV.DESTRU.                    615
VIV.AVER.                      218
VIAS                           564
ACUED.                         619
OTROS                          552
COMENTARIOS                      6
VALOR.1                          0
FECHA_año                        0
FECHA_mes                        0
dtype: int64


Como se puede observar con en la reducción del dataset en cuanto a la filtración y extracción de las columnas a usar, nos damos cuenta que 5 de las 11 columnas poseen nulos, pero el más importante es otros y comentarios, ¿Por qué? Esto porque en estas columnas podemos extraer la información si tuvo inplicaciones en daños de cultivos que es el objetivo principal de la investigación, entonces podría reducirce aún más los datos.



In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_24)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS.1,FAMILIAS.1,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
2467,2024-02-07,19809.0,CAUCA,TIMBIQUI,INUNDACION,2468.0,2611.0,629.0,NaN,NaN,19.0,NaN,NaN,"224 VIVIENDAS EN RIESGO, PÉRDIDAS DE CULTIVOS.","CDGRD CAUCA, INFORMA \nMUNICIPIO: TIMBIQUÍ, RE...",0.0,2024.0,2.0
4341,2024-03-13,27430.0,CHOCO,MEDIO BAUDO,INUNDACION,4342.0,12500.0,4050.0,760.0,7,NaN,NaN,NaN,912 AVES DE CORRAL Y 1560 EN PORCINOS,CDGRD CHOCÓ INFORMA: MUNICIPIO: MEDIO BAUDÓ ...,0.0,2024.0,3.0
4351,2024-03-13,27025.0,CHOCO,ALTO BAUDO,INUNDACION,4352.0,15000.0,7500.0,23000.0,35,7500.0,NaN,10,250.000 GALLINAS. ACCIONES:,CMGRD INFORMA: MUNICIPIO: ALTO BAUDÓ CHOCÓ/ ...,0.0,2024.0,3.0
5206,2024-04-01,19809.0,CAUCA,TIMBIQUI,INUNDACION,5207.0,3391.0,980.0,633.0,4,121.0,NaN,2,"431 ANIMALES, 11 COMUNIDADES AFECTADAS, 38 VIV...",CDGRD CAUCA INFORMA: MUNICIPIO: TIMBIQUÍ - CA...,0.0,2024.0,4.0
5303,2024-04-05,5147.0,ANTIOQUIA,CAREPA,INUNDACION,5304.0,7500.0,1875.0,270.0,NaN,NaN,NaN,NaN,"HECTÁREAS AFECTADAS DE CULTIVOS AFECTADOS, PLÁ...","DAGRAN ANTIOQUIA, INFORMA: MUNICIPIO: CAREPA, ...",0.0,2024.0,4.0


In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"PERSONAS.1","PERSONAS")
df_filtrado = cambiar_nombre_columna(df_filtrado,"FAMILIAS.1","FAMILIAS")

In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 124 entries, 2467 to 10256
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        124 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  124 non-null    float64       
 2   DEPARTAMENTO                 124 non-null    str           
 3   MUNICIPIO                    124 non-null    str           
 4   EVENTO                       124 non-null    str           
 5   CODIGO EMERGENCIA            124 non-null    float64       
 6   PERSONAS                     104 non-null    float64       
 7   FAMILIAS                     105 non-null    float64       
 8   HECTAREAS                    40 non-null     float64       
 9   VIV.DESTRU.                  11 non-null     object        
 10  VIV.AVER.                    71 non-null     float64       
 11  VIAS                         20 non-null     float64    

Como podemos ver, al realizar la filtración de los datos que poseen cultivos que normalmente son perdidas por inundaciones y/o deslizamientos se reduce considerablemente, aunque siendo quisquillosos podríamos sacar y filtrar aún más los datos, el problema es recaer en poca información, ¿Por qué no solo cultivos? Sería coherente pues no nos interesa si se afectaron vías, viviendas u otras cuestiones, sin embargo, esto puede ayudar a tener una mayor perspectiva a nivel economico del lugar y cuanta accesibilidad haya para adquirir un seguro.

Procedemos a guardar el dataset para acumularlo con los siguientes.

In [ ]:
df_definitivo = df_filtrado.copy()

In [ ]:
df_filtrado.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
2467,2024-02-07,19809.0,CAUCA,TIMBIQUI,INUNDACION,2468.0,2611.0,629.0,NaN,NaN,19.0,NaN,NaN,"224 VIVIENDAS EN RIESGO, PÉRDIDAS DE CULTIVOS.","CDGRD CAUCA, INFORMA \nMUNICIPIO: TIMBIQUÍ, RE...",0.0,2024.0,2.0
4341,2024-03-13,27430.0,CHOCO,MEDIO BAUDO,INUNDACION,4342.0,12500.0,4050.0,760.0,7,NaN,NaN,NaN,912 AVES DE CORRAL Y 1560 EN PORCINOS,CDGRD CHOCÓ INFORMA: MUNICIPIO: MEDIO BAUDÓ ...,0.0,2024.0,3.0
4351,2024-03-13,27025.0,CHOCO,ALTO BAUDO,INUNDACION,4352.0,15000.0,7500.0,23000.0,35,7500.0,NaN,10,250.000 GALLINAS. ACCIONES:,CMGRD INFORMA: MUNICIPIO: ALTO BAUDÓ CHOCÓ/ ...,0.0,2024.0,3.0
5206,2024-04-01,19809.0,CAUCA,TIMBIQUI,INUNDACION,5207.0,3391.0,980.0,633.0,4,121.0,NaN,2,"431 ANIMALES, 11 COMUNIDADES AFECTADAS, 38 VIV...",CDGRD CAUCA INFORMA: MUNICIPIO: TIMBIQUÍ - CA...,0.0,2024.0,4.0
5303,2024-04-05,5147.0,ANTIOQUIA,CAREPA,INUNDACION,5304.0,7500.0,1875.0,270.0,NaN,NaN,NaN,NaN,"HECTÁREAS AFECTADAS DE CULTIVOS AFECTADOS, PLÁ...","DAGRAN ANTIOQUIA, INFORMA: MUNICIPIO: CAREPA, ...",0.0,2024.0,4.0


#### 2023

##### Exploración

In [ ]:
df_2023 = obtener_datos("../Data/UNGRD/REPORTE-ANUAL-EMERGENCIAS-2023_REPARADO.xlsx")

In [ ]:
info_dataset(df_2023)

<class 'pandas.DataFrame'>
RangeIndex: 5719 entries, 0 to 5718
Data columns (total 83 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0                                1 non-null      datetime64[us]
 1   ..                           1 non-null      str           
 2   …                            1 non-null      str           
 3   FECHA                        5154 non-null   datetime64[us]
 4   DEPARTAMENTO                 5154 non-null   str           
 5   MUNICIPIO                    5154 non-null   str           
 6   EVENTO                       5152 non-null   str           
 7   CODIFICACIÓN SEGUN DIVIPOLA  5153 non-null   float64       
 8   FAMILIAS                     78 non-null     float64       
 9   PERSONAS                     78 non-null     float64       
 10  MUERTOS                      198 non-null    object        
 11  HERIDOS                      240 non-null    float64  

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "CODIFICACIÓN SEGUN DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS.1",
           "FAMILIAS.1", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1", "PERSONAS", "FAMILIAS"]
# Se extrae
df_23 = extraer_columnas(df_2023, nom_col)

In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_23)

<class 'pandas.DataFrame'>
RangeIndex: 5719 entries, 0 to 5718
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        5154 non-null   datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  5153 non-null   float64       
 2   DEPARTAMENTO                 5154 non-null   str           
 3   MUNICIPIO                    5154 non-null   str           
 4   EVENTO                       5152 non-null   str           
 5   CODIGO EMERGENCIA            5154 non-null   float64       
 6   PERSONAS.1                   1626 non-null   float64       
 7   FAMILIAS.1                   1494 non-null   float64       
 8   HECTAREAS                    2110 non-null   float64       
 9   VIV.DESTRU.                  289 non-null    float64       
 10  VIV.AVER.                    1282 non-null   float64       
 11  VIAS                         575 non-null    float64  

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
# df_23 = extraer_año_mes(df_23,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_23.copy()
df_23 = filtrar_datos(df_23,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_23)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 439 entries, 27 to 5125
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        439 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  438 non-null    float64       
 2   DEPARTAMENTO                 439 non-null    str           
 3   MUNICIPIO                    439 non-null    str           
 4   EVENTO                       439 non-null    str           
 5   CODIGO EMERGENCIA            439 non-null    float64       
 6   PERSONAS.1                   339 non-null    float64       
 7   FAMILIAS.1                   348 non-null    float64       
 8   HECTAREAS                    32 non-null     float64       
 9   VIV.DESTRU.                  45 non-null     float64       
 10  VIV.AVER.                    322 non-null    float64       
 11  VIAS                         61 

Como se puede observar, tiene la misma tendencia al dataset 2024 que no posee datos de deslizamientos.

In [ ]:
# Borramos la fehca
# df_23 = borrar_columnas(df_23,"FECHA")

In [ ]:
info_dataset(df_23)

<class 'pandas.DataFrame'>
Index: 439 entries, 27 to 5125
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        439 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  438 non-null    float64       
 2   DEPARTAMENTO                 439 non-null    str           
 3   MUNICIPIO                    439 non-null    str           
 4   EVENTO                       439 non-null    str           
 5   CODIGO EMERGENCIA            439 non-null    float64       
 6   PERSONAS.1                   339 non-null    float64       
 7   FAMILIAS.1                   348 non-null    float64       
 8   HECTAREAS                    32 non-null     float64       
 9   VIV.DESTRU.                  45 non-null     float64       
 10  VIV.AVER.                    322 non-null    float64       
 11  VIAS                         61 non-null     float64       

Como se puede observar hay menos observaciones, que el dataset 2024 por lo que tenemos indicios de que diversidad de información.

##### Segunda exploración de datos

In [ ]:
CO = ["PERSONAS","FAMILIAS"]
df_23 = borrar_columnas(df_23,CO)

In [ ]:
datos_nulos(df_23)

FECHA                            0
CODIFICACIÓN SEGUN DIVIPOLA      1
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                0
PERSONAS.1                     100
FAMILIAS.1                      91
HECTAREAS                      407
VIV.DESTRU.                    394
VIV.AVER.                      117
VIAS                           378
ACUED.                         415
OTROS                          390
COMENTARIOS                     46
VALOR.1                        437
dtype: int64


Misma observación que el anterior dataset, pero teniendo en cuenta que tanto comentarios y observaciones nos sirve para agarrar cualquier registro con información de cultivos.

In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_23)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS.1,FAMILIAS.1,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1
186,2023-01-12,73504.0,TOLIMA,ORTEGA,INUNDACION,187.0,64.0,16.0,19.0,NaN,16.0,NaN,NaN,DAÑOS EN CULTIVOS DE PANCOGER,CDGRD TOLIMA INFORMA: MUNICIPIO: ORTEGA - VERE...,NaN
191,2023-01-12,73504.0,TOLIMA,ORTEGA,INUNDACION,192.0,37.0,12.0,10.0,NaN,NaN,NaN,NaN,CULTIVO CACHACO Y MAÍZ,"CDGRD TOLIMA, INFORMA: MUNICIPIO ORTEGA, VUELT...",NaN
209,2023-01-13,52320.0,NARIÑO,GUAITARILLA,INUNDACION,210.0,752.0,221.0,NaN,12.0,174.0,6.0,3.0,"35 VIVIENDAS EN RIESGO, CULTIVOS","CDGRD NARIÑO, INFORMA\nMUNICIPIO GUAITARILLA, ...",NaN
223,2023-01-14,17380.0,CALDAS,LA DORADA,INUNDACION,224.0,200.0,50.0,NaN,NaN,50.0,NaN,NaN,NaN,CDGRD CALDAS INFORMA:\nMUNICIPIO LA DORADA - Z...,NaN
234,2023-01-14,15572.0,BOYACA,PUERTO BOYACA,INUNDACION,235.0,280.0,70.0,NaN,NaN,70.0,NaN,NaN,NaN,CDGRD BOYACÁ INFORMA:\nMUNICIPIO PUERTO BOYACÁ...,NaN


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 69 entries, 186 to 5090
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        69 non-null     datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  69 non-null     float64       
 2   DEPARTAMENTO                 69 non-null     str           
 3   MUNICIPIO                    69 non-null     str           
 4   EVENTO                       69 non-null     str           
 5   CODIGO EMERGENCIA            69 non-null     float64       
 6   PERSONAS.1                   60 non-null     float64       
 7   FAMILIAS.1                   63 non-null     float64       
 8   HECTAREAS                    29 non-null     float64       
 9   VIV.DESTRU.                  11 non-null     float64       
 10  VIV.AVER.                    54 non-null     float64       
 11  VIAS                         24 non-null     float64       

##### Unión de datasets

Cocatenamos los datos del dataset 24 y el 23 para formar uno y así guardar formar el deseado.

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"PERSONAS.1","PERSONAS")
df_filtrado = cambiar_nombre_columna(df_filtrado,"FAMILIAS.1","FAMILIAS")

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        193 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  193 non-null    float64       
 2   DEPARTAMENTO                 193 non-null    str           
 3   MUNICIPIO                    193 non-null    str           
 4   EVENTO                       193 non-null    str           
 5   CODIGO EMERGENCIA            193 non-null    float64       
 6   PERSONAS                     164 non-null    float64       
 7   FAMILIAS                     168 non-null    float64       
 8   HECTAREAS                    69 non-null     float64       
 9   VIV.DESTRU.                  22 non-null     object        
 10  VIV.AVER.                    125 non-null    float64       
 11  VIAS                         44 non-null     float64    

#### 2022

##### Exploración

In [ ]:
df_2022 = obtener_datos_para_modificados("../Data/UNGRD/REPORTE-ANUAL-DE-EMERGENCIAS-2022_REPARADO.xlsx")

In [ ]:
info_dataset(df_2022)

<class 'pandas.DataFrame'>
RangeIndex: 4579 entries, 0 to 4578
Data columns (total 100 columns):
 #   Column                                                 Non-Null Count  Dtype         
---  ------                                                 --------------  -----         
 0                                                          0 non-null      float64       
 1   ..                                                     0 non-null      float64       
 2   …                                                      0 non-null      float64       
 3   FECHA                                                  4555 non-null   datetime64[us]
 4   DEPARTAMENTO                                           4555 non-null   str           
 5   MUNICIPIO                                              4555 non-null   str           
 6   EVENTO                                                 4555 non-null   str           
 7   CODIFICACIÓN SEGUN DIVIPOLA                            4548 non-null   float64 

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "CODIFICACIÓN SEGUN DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1",]
# Se extrae
df_22 = extraer_columnas(df_2022, nom_col)

In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_22)

<class 'pandas.DataFrame'>
RangeIndex: 4579 entries, 0 to 4578
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        4555 non-null   datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  4548 non-null   float64       
 2   DEPARTAMENTO                 4555 non-null   str           
 3   MUNICIPIO                    4555 non-null   str           
 4   EVENTO                       4555 non-null   str           
 5   CODIGO EMERGENCIA            4546 non-null   float64       
 6   PERSONAS                     2288 non-null   float64       
 7   FAMILIAS                     2233 non-null   float64       
 8   HECTAREAS                    997 non-null    float64       
 9   VIV.DESTRU.                  437 non-null    float64       
 10  VIV.AVER.                    1864 non-null   float64       
 11  VIAS                         1146 non-null   float64  

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_22 = extraer_año_mes(df_22,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_22.copy()
df_22 = filtrar_datos(df_22,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_22)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 932 entries, 52 to 4541
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        932 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  932 non-null    float64       
 2   DEPARTAMENTO                 932 non-null    str           
 3   MUNICIPIO                    932 non-null    str           
 4   EVENTO                       932 non-null    str           
 5   CODIGO EMERGENCIA            932 non-null    float64       
 6   PERSONAS                     690 non-null    float64       
 7   FAMILIAS                     735 non-null    float64       
 8   HECTAREAS                    77 non-null     float64       
 9   VIV.DESTRU.                  71 non-null     float64       
 10  VIV.AVER.                    645 non-null    float64       
 11  VIAS                         123

Como se puede ver este dataset presentó algunos problemas al momento de la extracción de los datos debido a: 1) La fuente de datos estaba encriptada, 2) En la sección de RUD (Registro único de Damnificados en Colombia) no llenarón datos lo cual tendríamos que prescindir de este.

Por ahora se maneja los datos así y seguimos con los demás datasets.

In [ ]:
# Borramos la fehca
# df_22 = borrar_columnas(df_22,"FECHA")

In [ ]:
info_dataset(df_22)

<class 'pandas.DataFrame'>
Index: 932 entries, 52 to 4541
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        932 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  932 non-null    float64       
 2   DEPARTAMENTO                 932 non-null    str           
 3   MUNICIPIO                    932 non-null    str           
 4   EVENTO                       932 non-null    str           
 5   CODIGO EMERGENCIA            932 non-null    float64       
 6   PERSONAS                     690 non-null    float64       
 7   FAMILIAS                     735 non-null    float64       
 8   HECTAREAS                    77 non-null     float64       
 9   VIV.DESTRU.                  71 non-null     float64       
 10  VIV.AVER.                    645 non-null    float64       
 11  VIAS                         123 non-null    float64       

##### Segundo análisis del dataset

En este caso y contrariamente a lo que se hizo con el dataset 2024 y 2023 no se eliminará las variables de personas y familais porque estas ya corresponden a la sección de AFECTADOS siendo las variables a cocatenar para formar un solo dataset.

Revisamos nulos

In [ ]:
datos_nulos(df_22)

FECHA                            0
CODIFICACIÓN SEGUN DIVIPOLA      0
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                0
PERSONAS                       242
FAMILIAS                       197
HECTAREAS                      855
VIV.DESTRU.                    861
VIV.AVER.                      287
VIAS                           809
ACUED.                         886
OTROS                          838
COMENTARIOS                      0
VALOR.1                        931
FECHA_año                        0
FECHA_mes                        0
dtype: int64


Viendo los nulos, me preocupa en gran medida la poca información que las entidades públicas llenan la información, esto hace más díficil la tarea, además que al momento de realizar un análisis obstaculizan el mismo. No podríamos saber los valores o costos que tuvo ese daño.

In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_22)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
524,2022-02-14,5837.0,ANTIOQUIA,TURBO,INUNDACION,525.0,59.0,30.0,NaN,NaN,30.0,1.0,NaN,NaN,"CDGRD ANTIOQUIA INFORMA QUE, EN TURBO, CORREGI...",NaN,2022.0,2.0
702,2022-02-25,73030.0,TOLIMA,AMBALEMA,INUNDACION,703.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CDGRD TOLIMA INFORMA EN EL MUNICIPIO DE AMBALE...,NaN,2022.0,2.0
758,2022-02-28,25368.0,CUNDINAMARCA,JERUSALEN,INUNDACION,759.0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,"\nCDGRD DE CUNDINAMARCA, INFORMA\nMUNICIPIO JE...",NaN,2022.0,2.0
780,2022-03-01,27150.0,CHOCO,CARMEN DEL DARIEN,INUNDACION,781.0,243.0,78.0,NaN,NaN,NaN,NaN,NaN,NaN,CDGRD CHOCÓ INFORMA EN EL MUNICIPIO CARMEN DEL...,NaN,2022.0,3.0
793,2022-03-01,27800.0,CHOCO,UNGUIA,INUNDACION,794.0,600.0,150.0,NaN,NaN,150.0,NaN,1.0,NaN,CDGRD CHOCÓ INFORMA EN EL MUNICIPIO UNGUIA TAN...,NaN,2022.0,3.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 162 entries, 524 to 4387
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        162 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  162 non-null    float64       
 2   DEPARTAMENTO                 162 non-null    str           
 3   MUNICIPIO                    162 non-null    str           
 4   EVENTO                       162 non-null    str           
 5   CODIGO EMERGENCIA            162 non-null    float64       
 6   PERSONAS                     119 non-null    float64       
 7   FAMILIAS                     140 non-null    float64       
 8   HECTAREAS                    58 non-null     float64       
 9   VIV.DESTRU.                  24 non-null     float64       
 10  VIV.AVER.                    100 non-null    float64       
 11  VIAS                         36 non-null     float64      

##### Unión de datasets

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"PERSONAS.1","PERSONAS")
df_filtrado = cambiar_nombre_columna(df_filtrado,"FAMILIAS.1","FAMILIAS")

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 355 entries, 0 to 354
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        355 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  355 non-null    float64       
 2   DEPARTAMENTO                 355 non-null    str           
 3   MUNICIPIO                    355 non-null    str           
 4   EVENTO                       355 non-null    str           
 5   CODIGO EMERGENCIA            355 non-null    float64       
 6   PERSONAS                     283 non-null    float64       
 7   FAMILIAS                     308 non-null    float64       
 8   HECTAREAS                    127 non-null    float64       
 9   VIV.DESTRU.                  46 non-null     object        
 10  VIV.AVER.                    225 non-null    float64       
 11  VIAS                         80 non-null     float64    

In [ ]:
df_definitivo.head(100)

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
0,2024-02-07,19809.0,CAUCA,TIMBIQUI,INUNDACION,2468.0,2611.0,629.0,NaN,NaN,19.0,NaN,NaN,"224 VIVIENDAS EN RIESGO, PÉRDIDAS DE CULTIVOS.","CDGRD CAUCA, INFORMA \nMUNICIPIO: TIMBIQUÍ, RE...",0.0,2024.0,2.0
1,2024-03-13,27430.0,CHOCO,MEDIO BAUDO,INUNDACION,4342.0,12500.0,4050.0,760.0,7,NaN,NaN,NaN,912 AVES DE CORRAL Y 1560 EN PORCINOS,CDGRD CHOCÓ INFORMA: MUNICIPIO: MEDIO BAUDÓ ...,0.0,2024.0,3.0
2,2024-03-13,27025.0,CHOCO,ALTO BAUDO,INUNDACION,4352.0,15000.0,7500.0,23000.0,35,7500.0,NaN,10,250.000 GALLINAS. ACCIONES:,CMGRD INFORMA: MUNICIPIO: ALTO BAUDÓ CHOCÓ/ ...,0.0,2024.0,3.0
3,2024-04-01,19809.0,CAUCA,TIMBIQUI,INUNDACION,5207.0,3391.0,980.0,633.0,4,121.0,NaN,2,"431 ANIMALES, 11 COMUNIDADES AFECTADAS, 38 VIV...",CDGRD CAUCA INFORMA: MUNICIPIO: TIMBIQUÍ - CA...,0.0,2024.0,4.0
4,2024-04-05,5147.0,ANTIOQUIA,CAREPA,INUNDACION,5304.0,7500.0,1875.0,270.0,NaN,NaN,NaN,NaN,"HECTÁREAS AFECTADAS DE CULTIVOS AFECTADOS, PLÁ...","DAGRAN ANTIOQUIA, INFORMA: MUNICIPIO: CAREPA, ...",0.0,2024.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2024-08-24,27430.0,CHOCO,MEDIO BAUDO,INUNDACION,7885.0,6400.0,1600.0,NaN,NaN,NaN,NaN,NaN,"VIVIENDAS DESTECHADAS, CULTIVOS DE PANCOGER (...",CDGRD INFORMA: MUNICIPIO: MEDIO BAUDÓ CHOCÓ / ...,0.0,2024.0,8.0
96,2024-08-26,27600.0,CHOCO,RIO QUITO,INUNDACION,7910.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,\nCDGRD INFORMA: MUNICIPIO: RÍO QUITO CHOCÓ/ C...,0.0,2024.0,8.0
97,2024-08-26,47720.0,MAGDALENA,SANTA BARBARA DE PINTO,INUNDACION,7919.0,100.0,20.0,NaN,NaN,15.0,NaN,NaN,NaN,DCC INFORMA:\nMUNICIPIO: SANTA BÁRBARA DE PI...,0.0,2024.0,8.0
98,2024-08-26,27425.0,CHOCO,MEDIO ATRATO,INUNDACION,7924.0,3430.0,780.0,900.0,NaN,780.0,NaN,4,"PANCOGER, MAÍZ, YUCA, Y ANIMALES COMO PARTE AV...",CDGRD CHOCÓ INFORMA: MUNICIPIO: MEDIO ATRATO -...,0.0,2024.0,8.0


In [ ]:
df_definitivo.tail(100)

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
255,2022-06-11,18205.0,CAQUETA,CURILLO,INUNDACION,2127.0,240.0,60.0,NaN,NaN,60.0,NaN,NaN,NaN,CDGRD CAQUETÁ INFORMA:\nMUNICIPIO CURILLO – SE...,NaN,2022.0,6.0
256,2022-06-15,5120.0,ANTIOQUIA,CACERES,INUNDACION,2165.0,2200.0,550.0,NaN,NaN,20.0,NaN,NaN,NaN,\n\n*CDGRD DE ANTIOQUIA INFORMA*\n*MUNICIPIO* ...,NaN,2022.0,6.0
257,2022-06-15,44279.0,LA GUAJIRA,FONSECA,INUNDACION,2168.0,36.0,7.0,NaN,NaN,7.0,NaN,NaN,NaN,CDGRD LA GUAJIRA INFORMA EN EL MUNICIPIO DE FO...,NaN,2022.0,6.0
258,2022-06-16,5250.0,ANTIOQUIA,EL BAGRE,INUNDACION,2174.0,1500.0,735.0,NaN,1.0,734.0,NaN,NaN,NaN,CDGRD ANTIOQUIA INFORMA EN EL MUNICIPIO EL BAG...,NaN,2022.0,6.0
259,2022-06-17,76275.0,VALLE DEL CAUCA,FLORIDA,INUNDACION,2182.0,50.0,10.0,NaN,NaN,10.0,NaN,NaN,CULTIVOS DE CAÑA,"\nCDGRD DEL VALLE DEL CAUCA ACTUALIZA, INFORMA...",NaN,2022.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350,2022-11-21,27086.0,CHOCO,BELEN DE BAJIRA,INUNDACION,4306.0,7912.0,1978.0,NaN,NaN,894.0,NaN,NaN,NaN,CDGRD CHOCÓ INFORMA: MUNICIPIO BELÉN DE BAJIRÁ...,NaN,2022.0,11.0
351,2022-12-26,5475.0,ANTIOQUIA,MURINDO,INUNDACION,4348.0,1856.0,420.0,NaN,NaN,NaN,NaN,NaN,NaN,CDGRD ANTIOQUIA INFORMA EN EL MUNICIPIO DE MUR...,NaN,2022.0,12.0
352,2022-11-08,13160.0,BOLIVAR,CANTAGALLO,INUNDACION,4374.0,1251.0,417.0,367.0,3.0,NaN,NaN,NaN,"7 EQUINOS, 41 PORCINOS Y 39 CAPRINOS",CDGRD BOLÍVAR INFORMA:\nMUNICIPIO CANTAGALLO ...,NaN,2022.0,11.0
353,2022-10-10,20621.0,CESAR,LA PAZ,INUNDACION,4387.0,127.0,30.0,200.0,NaN,30.0,NaN,NaN,"DE CULTIVO, PÉRDIDA DE ANIMALES (5458 GALLINAS...","CDGRD CESAR, INFORMA: MUNICIPIO LA PAZ, VEREDA...",NaN,2022.0,10.0


#### 2021

##### Exploración

Por lo que se observa e intuye los datasets están encriptados, ergo provoca mayor dificultad para extraer datos y realizar un análisis.

Por otro lado, a partir del dataset 2022 estos conjuntos de datos no tienen la misma estructura al 100% que del 2024 y 2023, estos en la sección del RUD no tiene información de las personas/familias afectadas, entoces hace que la dinamica cambie.

In [ ]:
df_2021 = obtener_datos_para_modificados("../Data/UNGRD/EMERGENCIAS-2021-DICIEMBRE_REPARADO.xlsx")

In [ ]:
info_dataset(df_2021)

<class 'pandas.DataFrame'>
RangeIndex: 3955 entries, 0 to 3954
Data columns (total 98 columns):
 #   Column                                                 Non-Null Count  Dtype         
---  ------                                                 --------------  -----         
 0                                                          1 non-null      str           
 1   ..                                                     0 non-null      float64       
 2   …                                                      0 non-null      float64       
 3   FECHA                                                  3945 non-null   datetime64[us]
 4   DEPARTAMENTO                                           3945 non-null   str           
 5   MUNICIPIO                                              3945 non-null   str           
 6   EVENTO                                                 3946 non-null   str           
 7   CODIFICACIÓN SEGUN DIVIPOLA                            3944 non-null   object   

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "CODIFICACIÓN SEGUN DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
df_21 = extraer_columnas(df_2021, nom_col)



In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_21)

<class 'pandas.DataFrame'>
RangeIndex: 3955 entries, 0 to 3954
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        3945 non-null   datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  3944 non-null   object        
 2   DEPARTAMENTO                 3945 non-null   str           
 3   MUNICIPIO                    3945 non-null   str           
 4   EVENTO                       3946 non-null   str           
 5   CODIGO EMERGENCIA            3945 non-null   float64       
 6   PERSONAS                     2095 non-null   float64       
 7   FAMILIAS                     1990 non-null   float64       
 8   HECTAREAS                    831 non-null    object        
 9   VIV.DESTRU.                  326 non-null    float64       
 10  VIV.AVER.                    1665 non-null   object        
 11  VIAS                         902 non-null    float64  

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_21 = extraer_año_mes(df_21,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_21.copy()
df_21 = filtrar_datos(df_21,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_21)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 802 entries, 2 to 3944
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        802 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  802 non-null    object        
 2   DEPARTAMENTO                 802 non-null    str           
 3   MUNICIPIO                    802 non-null    str           
 4   EVENTO                       802 non-null    str           
 5   CODIGO EMERGENCIA            802 non-null    float64       
 6   PERSONAS                     600 non-null    float64       
 7   FAMILIAS                     625 non-null    float64       
 8   HECTAREAS                    75 non-null     object        
 9   VIV.DESTRU.                  53 non-null     float64       
 10  VIV.AVER.                    540 non-null    object        
 11  VIAS                         109 

Dado la diferencia y semejanzas en ciertas cuestiones, se contempla revisar desde el Excel que puede contener.

Actualizado: Se puede observar que los datos de inundaciones no siempre contemplas cultivos, por no decir que es un información que aparece poco, por lo que al filtrar aún más lo datos se reduciría el dataset.

In [ ]:
# Borramos la fehca
# df_21 = borrar_columnas(df_21,"FECHA")

In [ ]:
info_dataset(df_21)

<class 'pandas.DataFrame'>
Index: 802 entries, 2 to 3944
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        802 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  802 non-null    object        
 2   DEPARTAMENTO                 802 non-null    str           
 3   MUNICIPIO                    802 non-null    str           
 4   EVENTO                       802 non-null    str           
 5   CODIGO EMERGENCIA            802 non-null    float64       
 6   PERSONAS                     600 non-null    float64       
 7   FAMILIAS                     625 non-null    float64       
 8   HECTAREAS                    75 non-null     object        
 9   VIV.DESTRU.                  53 non-null     float64       
 10  VIV.AVER.                    540 non-null    object        
 11  VIAS                         109 non-null    float64       


##### Segundo análisis y preparación del dataset

In [ ]:
datos_nulos(df_21)

FECHA                            0
CODIFICACIÓN SEGUN DIVIPOLA      0
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                0
PERSONAS                       202
FAMILIAS                       177
HECTAREAS                      727
VIV.DESTRU.                    749
VIV.AVER.                      262
VIAS                           693
ACUED.                         757
OTROS                          698
COMENTARIOS                      3
VALOR.1                          1
FECHA_año                        0
FECHA_mes                        0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_21)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
102,2021-01-15,27150,CHOCO,CARMEN DEL DARIEN,INUNDACION,103.0,9095.0,1819.0,NaN,NaN,NaN,NaN,NaN,"PÉRDIDAS DE ENSERES, CULTIVOS DE PLÁTANO, YUCA...","\nENLACE DEL CHOCÓ- UNGRD, INFORMA\nMUNICIPIO ...",0.0,2021.0,1.0
160,2021-01-21,27430,CHOCO,MEDIO BAUDO,INUNDACION,161.0,13956.0,3489.0,899,40.0,NaN,3.0,23.0,CULTIVOS AGRICOLAS Y DE PANCOGER,CMGRD MEDIO BAUDÓ – CHOCÓ INFORMA\nMUNICIPIO: ...,0.0,2021.0,1.0
169,2021-01-23,27150,CHOCO,CARMEN DEL DARIEN,INUNDACION,170.0,847.0,172.0,NaN,NaN,NaN,NaN,NaN,NaN,ENLACE UNGRD CHOCO INFORMA\nMUNICIPIO: CARMEN ...,0.0,2021.0,1.0
390,2021-02-07,19780,CAUCA,SUAREZ,INUNDACION,391.0,5.0,1.0,NaN,NaN,1,NaN,NaN,1 HECTÁREA DE CULTIVOS DE PANCOGER,"CDGRD CAUCA INFORMA QUE, EN SUÁREZ, CORREGIMIE...",0.0,2021.0,2.0
749,2021-03-08,05585,ANTIOQUIA,PUERTO NARE,INUNDACION,750.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CDGRD ANTIOQUIA INFORMA MUNICIPIO: PUERTO NARE...,0.0,2021.0,3.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 150 entries, 102 to 3905
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        150 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  150 non-null    object        
 2   DEPARTAMENTO                 150 non-null    str           
 3   MUNICIPIO                    150 non-null    str           
 4   EVENTO                       150 non-null    str           
 5   CODIGO EMERGENCIA            150 non-null    float64       
 6   PERSONAS                     110 non-null    float64       
 7   FAMILIAS                     119 non-null    float64       
 8   HECTAREAS                    61 non-null     object        
 9   VIV.DESTRU.                  15 non-null     float64       
 10  VIV.AVER.                    86 non-null     object        
 11  VIAS                         36 non-null     float64      

##### Unión de los datasets

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        505 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  505 non-null    object        
 2   DEPARTAMENTO                 505 non-null    str           
 3   MUNICIPIO                    505 non-null    str           
 4   EVENTO                       505 non-null    str           
 5   CODIGO EMERGENCIA            505 non-null    float64       
 6   PERSONAS                     393 non-null    float64       
 7   FAMILIAS                     427 non-null    float64       
 8   HECTAREAS                    188 non-null    object        
 9   VIV.DESTRU.                  61 non-null     object        
 10  VIV.AVER.                    311 non-null    object        
 11  VIAS                         116 non-null    float64    

#### 2020

##### Exploración

In [ ]:
df_2020 = obtener_datos_para_modificados("../Data/UNGRD/EMERGENCIAS-2020_REPARADO.xlsx")

In [ ]:
info_dataset(df_2020)

<class 'pandas.DataFrame'>
RangeIndex: 3849 entries, 0 to 3848
Data columns (total 98 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   QSQ                                                    0 non-null      float64
 1   ..                                                     0 non-null      float64
 2   …                                                      0 non-null      float64
 3   FECHA                                                  3811 non-null   object 
 4   DEPARTAMENTO                                           3811 non-null   str    
 5   MUNICIPIO                                              3811 non-null   str    
 6   EVENTO                                                 3804 non-null   str    
 7   CODIFICACIÓN SEGUN DIVIPOLA                            3810 non-null   object 
 8   RUD                                                    0 no

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "CODIFICACIÓN SEGUN DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_20 = extraer_columnas(df_2020, nom_col)

In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_20)

<class 'pandas.DataFrame'>
RangeIndex: 3849 entries, 0 to 3848
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   FECHA                        3811 non-null   object 
 1   CODIFICACIÓN SEGUN DIVIPOLA  3810 non-null   object 
 2   DEPARTAMENTO                 3811 non-null   str    
 3   MUNICIPIO                    3811 non-null   str    
 4   EVENTO                       3804 non-null   str    
 5   CODIGO EMERGENCIA            3807 non-null   float64
 6   PERSONAS                     1491 non-null   float64
 7   FAMILIAS                     1311 non-null   float64
 8   HECTAREAS                    1526 non-null   object 
 9   VIV.DESTRU.                  242 non-null    float64
 10  VIV.AVER.                    1098 non-null   float64
 11  VIAS                         342 non-null    float64
 12  ACUED.                       68 non-null     float64
 13  OTROS                        

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_20['FECHA'] = pd.to_datetime(df_20['FECHA'], errors='coerce')
df_20 = extraer_año_mes(df_20,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_20.copy()
df_20 = filtrar_datos(df_20,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_20)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 428 entries, 176 to 3810
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        427 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  428 non-null    object        
 2   DEPARTAMENTO                 428 non-null    str           
 3   MUNICIPIO                    428 non-null    str           
 4   EVENTO                       428 non-null    str           
 5   CODIGO EMERGENCIA            425 non-null    float64       
 6   PERSONAS                     329 non-null    float64       
 7   FAMILIAS                     337 non-null    float64       
 8   HECTAREAS                    25 non-null     object        
 9   VIV.DESTRU.                  32 non-null     float64       
 10  VIV.AVER.                    294 non-null    float64       
 11  VIAS                         45

In [ ]:
# Borramos la fehca
# df_20 = borrar_columnas(df_20,"FECHA")

In [ ]:
info_dataset(df_20)

<class 'pandas.DataFrame'>
Index: 428 entries, 176 to 3810
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        427 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  428 non-null    object        
 2   DEPARTAMENTO                 428 non-null    str           
 3   MUNICIPIO                    428 non-null    str           
 4   EVENTO                       428 non-null    str           
 5   CODIGO EMERGENCIA            425 non-null    float64       
 6   PERSONAS                     329 non-null    float64       
 7   FAMILIAS                     337 non-null    float64       
 8   HECTAREAS                    25 non-null     object        
 9   VIV.DESTRU.                  32 non-null     float64       
 10  VIV.AVER.                    294 non-null    float64       
 11  VIAS                         45 non-null     float64      

##### Segundo análisis y preparación

In [ ]:
datos_nulos(df_20)

FECHA                            1
CODIFICACIÓN SEGUN DIVIPOLA      0
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                3
PERSONAS                        99
FAMILIAS                        91
HECTAREAS                      403
VIV.DESTRU.                    396
VIV.AVER.                      134
VIAS                           383
ACUED.                         415
OTROS                          372
COMENTARIOS                      0
VALOR.1                          1
FECHA_año                        1
FECHA_mes                        1
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_20)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
176,2020-06-09,70110,SUCRE,BUENAVISTA,INUNDACION,179.0,190.0,38.0,NaN,NaN,38.0,2.0,NaN,NaN,CDGRD SUCRE INFORMA EN EL MUNICIPIO DE BUENAVI...,0.0,2020.0,6.0
380,2020-02-02,27430,CHOCO,MEDIO BAUDO,INUNDACION,383.0,NaN,612.0,NaN,NaN,NaN,NaN,NaN,NaN,"CDGRD CHOCÓ INFORMA QUE, EN MEDIO BAUDÓ, ZONA ...",0.0,2020.0,2.0
848,2020-09-30,70124,SUCRE,CAIMITO,INUNDACION,851.0,5.0,1.0,NaN,NaN,1.0,NaN,NaN,NaN,"CDGRD DE SUCRE, INFORMA\nMUNICIPIO CAIMITO, VE...",0.0,2020.0,9.0
1367,2020-08-19,70221,SUCRE,COVEÑAS,INUNDACION,1371.0,1612.0,403.0,NaN,NaN,403.0,NaN,NaN,NaN,CDGRD SUCRE INFORMA EN EL MUNICIPIO COVEÑAS IS...,0.0,2020.0,8.0
1389,2020-04-01,27099,CHOCO,BOJAYA,INUNDACION,1393.0,1390.0,278.0,720,NaN,61.0,NaN,NaN,NaN,"ENLACE UNGRD CHOCÓ INFORMA QUE, EN BOJACÁ, COR...",0.0,2020.0,4.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 97 entries, 176 to 3804
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        97 non-null     datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  97 non-null     object        
 2   DEPARTAMENTO                 97 non-null     str           
 3   MUNICIPIO                    97 non-null     str           
 4   EVENTO                       97 non-null     str           
 5   CODIGO EMERGENCIA            96 non-null     float64       
 6   PERSONAS                     75 non-null     float64       
 7   FAMILIAS                     81 non-null     float64       
 8   HECTAREAS                    22 non-null     object        
 9   VIV.DESTRU.                  10 non-null     float64       
 10  VIV.AVER.                    55 non-null     float64       
 11  VIAS                         17 non-null     float64       

##### Unión de los datasets

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 602 entries, 0 to 601
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        602 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  602 non-null    object        
 2   DEPARTAMENTO                 602 non-null    str           
 3   MUNICIPIO                    602 non-null    str           
 4   EVENTO                       602 non-null    str           
 5   CODIGO EMERGENCIA            601 non-null    float64       
 6   PERSONAS                     468 non-null    float64       
 7   FAMILIAS                     508 non-null    float64       
 8   HECTAREAS                    210 non-null    object        
 9   VIV.DESTRU.                  71 non-null     object        
 10  VIV.AVER.                    366 non-null    object        
 11  VIAS                         133 non-null    float64    

#### 2019

##### Exploración

In [ ]:
df_2019 = obtener_datos_para_modificados("../Data/UNGRD/EMERGENCIAS-2019_REPARADO.xlsx")

In [ ]:
info_dataset(df_2019)

<class 'pandas.DataFrame'>
RangeIndex: 4461 entries, 0 to 4460
Data columns (total 98 columns):
 #   Column                                                 Non-Null Count  Dtype         
---  ------                                                 --------------  -----         
 0   QSQ                                                    5 non-null      object        
 1   ..                                                     0 non-null      float64       
 2   …                                                      0 non-null      float64       
 3   FECHA                                                  4436 non-null   datetime64[us]
 4   DEPARTAMENTO                                           4436 non-null   str           
 5   MUNICIPIO                                              4436 non-null   str           
 6   EVENTO                                                 4436 non-null   str           
 7   CODIFICACIÓN SEGUN DIVIPOLA                            2 non-null      float64  

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "CODIFICACIÓN SEGUN DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_19 = extraer_columnas(df_2019, nom_col)

In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_19)

<class 'pandas.DataFrame'>
RangeIndex: 4461 entries, 0 to 4460
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        4436 non-null   datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  2 non-null      float64       
 2   DEPARTAMENTO                 4436 non-null   str           
 3   MUNICIPIO                    4436 non-null   str           
 4   EVENTO                       4436 non-null   str           
 5   CODIGO EMERGENCIA            4432 non-null   float64       
 6   PERSONAS                     1357 non-null   float64       
 7   FAMILIAS                     1068 non-null   float64       
 8   HECTAREAS                    2118 non-null   object        
 9   VIV.DESTRU.                  280 non-null    float64       
 10  VIV.AVER.                    870 non-null    float64       
 11  VIAS                         325 non-null    float64  

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_19 = extraer_año_mes(df_19,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_19.copy()
df_19 = filtrar_datos(df_19,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_19)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 404 entries, 134 to 4435
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        404 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  2 non-null      float64       
 2   DEPARTAMENTO                 404 non-null    str           
 3   MUNICIPIO                    404 non-null    str           
 4   EVENTO                       404 non-null    str           
 5   CODIGO EMERGENCIA            402 non-null    float64       
 6   PERSONAS                     262 non-null    float64       
 7   FAMILIAS                     267 non-null    float64       
 8   HECTAREAS                    19 non-null     object        
 9   VIV.DESTRU.                  29 non-null     float64       
 10  VIV.AVER.                    229 non-null    float64       
 11  VIAS                         45

In [ ]:
# Borramos la fehca
#df_19 = borrar_columnas(df_19,"FECHA")

In [ ]:
info_dataset(df_19)

<class 'pandas.DataFrame'>
Index: 404 entries, 134 to 4435
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        404 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  2 non-null      float64       
 2   DEPARTAMENTO                 404 non-null    str           
 3   MUNICIPIO                    404 non-null    str           
 4   EVENTO                       404 non-null    str           
 5   CODIGO EMERGENCIA            402 non-null    float64       
 6   PERSONAS                     262 non-null    float64       
 7   FAMILIAS                     267 non-null    float64       
 8   HECTAREAS                    19 non-null     object        
 9   VIV.DESTRU.                  29 non-null     float64       
 10  VIV.AVER.                    229 non-null    float64       
 11  VIAS                         45 non-null     float64      

##### Segundo análisis y preparación

In [ ]:
print("FILAS/COLUMNAS",df_19.shape)
datos_nulos(df_19)

FECHA                            0
CODIFICACIÓN SEGUN DIVIPOLA    402
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                2
PERSONAS                       142
FAMILIAS                       137
HECTAREAS                      385
VIV.DESTRU.                    375
VIV.AVER.                      175
VIAS                           359
ACUED.                         382
OTROS                          379
COMENTARIOS                      0
VALOR.1                          0
FECHA_año                        0
FECHA_mes                        0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_19)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
885,2019-02-24,NaN,HUILA,GARZON,INUNDACION,887.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,*CDGRD HUILA Y CRUZ ROJA INFORMAN* *MUNICIPIO:...,0.0,2019.0,2.0
1144,2019-03-14,NaN,RISARALDA,SANTA ROSA DE CABAL,INUNDACION,1146.0,60.0,12.0,NaN,NaN,12.0,1.0,1.0,NaN,"CDGRD DE RISARALDA, INFORMA MUNICIPIO: SANTA R...",0.0,2019.0,3.0
1438,2019-04-05,NaN,CAUCA,CORINTO,INUNDACION,1440.0,319.0,107.0,NaN,7.0,40.0,9.0,11.0,NaN,*CDGRD CAUCA INFORMA* *MUNICIPIO:* CORINTO *E...,0.0,2019.0,4.0
1631,2019-04-19,NaN,CAUCA,LOPEZ DE MICAY,INUNDACION,1634.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CDGRD CAUCA INFORMA EN EL MUNICIPIO DE LÓPEZ D...,0.0,2019.0,4.0
1678,2019-04-22,NaN,ANTIOQUIA,CHIGORODO,INUNDACION,1681.0,200.0,40.0,NaN,NaN,40.0,NaN,NaN,NaN,"CDGRD DE ANTIOQUIA, INFORMA MUNICIPIO: CHIGORO...",0.0,2019.0,4.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 61 entries, 885 to 4435
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        61 non-null     datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  1 non-null      float64       
 2   DEPARTAMENTO                 61 non-null     str           
 3   MUNICIPIO                    61 non-null     str           
 4   EVENTO                       61 non-null     str           
 5   CODIGO EMERGENCIA            60 non-null     float64       
 6   PERSONAS                     38 non-null     float64       
 7   FAMILIAS                     42 non-null     float64       
 8   HECTAREAS                    10 non-null     object        
 9   VIV.DESTRU.                  9 non-null      float64       
 10  VIV.AVER.                    33 non-null     float64       
 11  VIAS                         16 non-null     float64       

##### Unión de los datasets

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 663 entries, 0 to 662
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        663 non-null    datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  603 non-null    object        
 2   DEPARTAMENTO                 663 non-null    str           
 3   MUNICIPIO                    663 non-null    str           
 4   EVENTO                       663 non-null    str           
 5   CODIGO EMERGENCIA            661 non-null    float64       
 6   PERSONAS                     506 non-null    float64       
 7   FAMILIAS                     550 non-null    float64       
 8   HECTAREAS                    220 non-null    object        
 9   VIV.DESTRU.                  80 non-null     object        
 10  VIV.AVER.                    399 non-null    object        
 11  VIAS                         149 non-null    float64    

#### 2018

##### Exploración

In [ ]:
df_2018 = obtener_datos_para_modificados("../Data/UNGRD/EMERGENCIAS-2018_REPARADO.xlsx")

In [ ]:
info_dataset(df_2018)

<class 'pandas.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Columns: 157 entries, . to Unnamed: 156
dtypes: float64(36), object(107), str(14)
memory usage: 6.0+ MB


In [ ]:
import math

cols = df_2018.columns.tolist()
total_cols = len(cols)
# Calculamos cuántas filas necesitamos (división hacia arriba)
filas = math.ceil(total_cols / 3) 

print(f"{'COLUMNA 1'.ljust(40)} | {'COLUMNA 2'.ljust(40)} | {'COLUMNA 3'}")
print("-" * 120)

for i in range(filas):
    col1 = cols[i] if i < total_cols else ""
    col2 = cols[i + filas] if (i + filas) < total_cols else ""
    col3 = cols[i + (filas * 2)] if (i + (filas * 2)) < total_cols else ""
    
    # Usamos ljust para que las tres columnas queden alineadas
    print(f"{col1.ljust(40)} | {col2.ljust(40)} | {col3}")

print("\n" + "-"*40)
print(f"Cantidad total de columnas: {total_cols}")

COLUMNA 1                                | COLUMNA 2                                | COLUMNA 3
------------------------------------------------------------------------------------------------------------------------
.                                        | CANTIDAD.2                               | VALOR.28
..                                       | VALOR.2                                  | CANTIDAD.29
…                                        | CANTIDAD.3                               | VALOR.29
FECHA                                    | VALOR.3                                  | CANTIDAD.30
DEPARTAMENTO                             | CANTIDAD.4                               | VALOR.30
MUNICIPIO                                | VALOR.4                                  | VALOR TOTAL
EVENTO                                   | CANTIDAD.5                               | CANTIDAD.31
CODIFICACIÓN SEGUN DIVIPOLA              | VALOR.5                                  | VALOR.31
RUD        

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "CODIFICACIÓN SEGUN DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_18 = extraer_columnas(df_2018, nom_col)

In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_18)

<class 'pandas.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   FECHA                        3495 non-null   object
 1   CODIFICACIÓN SEGUN DIVIPOLA  3487 non-null   object
 2   DEPARTAMENTO                 3496 non-null   str   
 3   MUNICIPIO                    3490 non-null   str   
 4   EVENTO                       3490 non-null   str   
 5   CODIGO EMERGENCIA            3490 non-null   object
 6   PERSONAS                     3122 non-null   object
 7   FAMILIAS                     3044 non-null   object
 8   HECTAREAS                    2856 non-null   object
 9   VIV.DESTRU.                  2796 non-null   object
 10  VIV.AVER.                    2973 non-null   object
 11  VIAS                         2819 non-null   object
 12  ACUED.                       2747 non-null   object
 13  OTROS                        2769 non-null  

In [ ]:
df_18.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1
0,2018-01-01 00:00:00,#REF!,BOYACA,GUATEQUE,INCENDIO FORESTAL,1,0,0,1,0,0,0,0,0,"CDGRD DE BOYACÁ INFORMA: FECHA: 01/01/2018, ...",0
1,2018-01-01 00:00:00,#REF!,CUNDINAMARCA,GUASCA,DESLIZAMIENTO,2,0,0,0,0,0,1,0,0,"CDGRD DE CUNDINAMARCA, INFORMA, SE PRESENTÓ DE...",0
2,2018-01-01 00:00:00,#REF!,QUINDIO,MONTENEGRO,DESLIZAMIENTO,3,1,0,0,0,0,1,0,0,CDGRD DE QUINDÍO REPORTA: MUNICIPIO: MONTENEG...,0
3,2018-01-02 00:00:00,#REF!,ANTIOQUIA,ANORI,DESLIZAMIENTO,4,15,3,0,1,2,0,0,0,CDGRD O/P DE ANTIOQUIA INFORMA: FECHA: 05/01/...,0
4,2018-01-02 00:00:00,#REF!,CHOCO,LLORO,ACCIDENTE FLUVIAL,5,2,0,0,0,0,0,0,0,"CDGRD CHOCÓ INFORMA: M/PIO. LLORÓ, HORA: ENTRE...",0


##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_18['FECHA'] = pd.to_datetime(df_18['FECHA'], errors='coerce')
df_18 = extraer_año_mes(df_18,"FECHA")

In [ ]:
df_18.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
0,2018-01-01,#REF!,BOYACA,GUATEQUE,INCENDIO FORESTAL,1,0,0,1,0,0,0,0,0,"CDGRD DE BOYACÁ INFORMA: FECHA: 01/01/2018, ...",0,2018.0,1.0
1,2018-01-01,#REF!,CUNDINAMARCA,GUASCA,DESLIZAMIENTO,2,0,0,0,0,0,1,0,0,"CDGRD DE CUNDINAMARCA, INFORMA, SE PRESENTÓ DE...",0,2018.0,1.0
2,2018-01-01,#REF!,QUINDIO,MONTENEGRO,DESLIZAMIENTO,3,1,0,0,0,0,1,0,0,CDGRD DE QUINDÍO REPORTA: MUNICIPIO: MONTENEG...,0,2018.0,1.0
3,2018-01-02,#REF!,ANTIOQUIA,ANORI,DESLIZAMIENTO,4,15,3,0,1,2,0,0,0,CDGRD O/P DE ANTIOQUIA INFORMA: FECHA: 05/01/...,0,2018.0,1.0
4,2018-01-02,#REF!,CHOCO,LLORO,ACCIDENTE FLUVIAL,5,2,0,0,0,0,0,0,0,"CDGRD CHOCÓ INFORMA: M/PIO. LLORÓ, HORA: ENTRE...",0,2018.0,1.0


In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_18.copy()
df_18 = filtrar_datos(df_18,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_18)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 37 entries, 69 to 3372
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        37 non-null     datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  37 non-null     object        
 2   DEPARTAMENTO                 37 non-null     str           
 3   MUNICIPIO                    37 non-null     str           
 4   EVENTO                       37 non-null     str           
 5   CODIGO EMERGENCIA            37 non-null     object        
 6   PERSONAS                     31 non-null     object        
 7   FAMILIAS                     32 non-null     object        
 8   HECTAREAS                    15 non-null     object        
 9   VIV.DESTRU.                  19 non-null     object        
 10  VIV.AVER.                    30 non-null     object        
 11  VIAS                         14 n

Dado que este es el primer dataset que hay aparecen registros de deslizamientos, realizamos la unión para no perder información.

In [ ]:
#df_18 = unir_datasets(df_18,df_copy)
# Se decide no unir los dataframes de deslizamientos con inundaciones para realizar el modelo con unicamente inundacionees

In [ ]:
# Borramos la fehca
#df_18 = borrar_columnas(df_18,"FECHA")

In [ ]:
info_dataset(df_18)

<class 'pandas.DataFrame'>
Index: 37 entries, 69 to 3372
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        37 non-null     datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  37 non-null     object        
 2   DEPARTAMENTO                 37 non-null     str           
 3   MUNICIPIO                    37 non-null     str           
 4   EVENTO                       37 non-null     str           
 5   CODIGO EMERGENCIA            37 non-null     object        
 6   PERSONAS                     31 non-null     object        
 7   FAMILIAS                     32 non-null     object        
 8   HECTAREAS                    15 non-null     object        
 9   VIV.DESTRU.                  19 non-null     object        
 10  VIV.AVER.                    30 non-null     object        
 11  VIAS                         14 non-null     object        


##### Segundo análisis y preparación

In [ ]:
datos_nulos(df_18)

FECHA                           0
CODIFICACIÓN SEGUN DIVIPOLA     0
DEPARTAMENTO                    0
MUNICIPIO                       0
EVENTO                          0
CODIGO EMERGENCIA               0
PERSONAS                        6
FAMILIAS                        5
HECTAREAS                      22
VIV.DESTRU.                    18
VIV.AVER.                       7
VIAS                           23
ACUED.                         22
OTROS                          23
COMENTARIOS                     0
VALOR.1                         0
FECHA_año                       0
FECHA_mes                       0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_18)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,CODIFICACIÓN SEGUN DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
582,2018-02-22,#REF!,CHOCO,LLORO,INUNDACION,583,543,417,0,0,0,1,1,0,CDGRD CHOCO REPORTA MUNICIPIO LLORO SE REGISTR...,0,2018.0,2.0
2490,2018-09-05,#REF!,CHOCO,CANTON DE SAN PABLO,INUNDACION,2491,0,0,0,0,0,0,0,0,EN ENLACE REALIZADO CON CMGRD CANTÓN DE SAN PA...,0,2018.0,9.0
2577,2018-09-13,#REF!,CHOCO,MEDIO SAN JUAN,INUNDACION,2578,1250,250,0,0,25,0,0,0,CDGRD CHOCO INFORMA EN EL MUNICIPIO DE MEDIO S...,0,2018.0,9.0
2818,2018-10-05,#REF!,CHOCO,NOVITA,INUNDACION,2819,2240,583,NaN,7,NaN,NaN,NaN,NaN,CDGRD CHOCO INFORMA EN EL MUNICIPIO DE NOVITA ...,0,2018.0,10.0
2819,2018-10-05,#REF!,CHOCO,SIPI,INUNDACION,2820,1896,647,NaN,56,35,NaN,NaN,NaN,"CDGRD DE CHOCÓ INFORMA: EVENTO: INUNDACION, MU...",0,2018.0,10.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 6 entries, 582 to 3060
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        6 non-null      datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  6 non-null      object        
 2   DEPARTAMENTO                 6 non-null      str           
 3   MUNICIPIO                    6 non-null      str           
 4   EVENTO                       6 non-null      str           
 5   CODIGO EMERGENCIA            6 non-null      object        
 6   PERSONAS                     6 non-null      object        
 7   FAMILIAS                     6 non-null      object        
 8   HECTAREAS                    3 non-null      object        
 9   VIV.DESTRU.                  5 non-null      object        
 10  VIV.AVER.                    4 non-null      object        
 11  VIAS                         3 non-null      object        


##### Unión de los datasets

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        669 non-null    datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  609 non-null    object        
 2   DEPARTAMENTO                 669 non-null    str           
 3   MUNICIPIO                    669 non-null    str           
 4   EVENTO                       669 non-null    str           
 5   CODIGO EMERGENCIA            667 non-null    object        
 6   PERSONAS                     512 non-null    object        
 7   FAMILIAS                     556 non-null    object        
 8   HECTAREAS                    223 non-null    object        
 9   VIV.DESTRU.                  85 non-null     object        
 10  VIV.AVER.                    403 non-null    object        
 11  VIAS                         152 non-null    object     

#### 2017

##### Exploración

In [ ]:
df_2017 = obtener_datos_para_modificados("../Data/UNGRD/EMERGENCIAS_2017_REPARADO.xlsx")

In [ ]:
info_dataset(df_2017)

<class 'pandas.DataFrame'>
RangeIndex: 3363 entries, 0 to 3362
Columns: 121 entries, FECHA to Unnamed: 120
dtypes: float64(99), object(12), str(10)
memory usage: 4.6+ MB


In [ ]:
df_2017.columns

Index(['FECHA', 'DEPARTAMENTO', 'MUNICIPIO', 'EVENTO',
       'Codificación según DIVIPOLA', 'MUERTOS', 'HERIDOS', 'DESAPA.',
       'PERSONAS', 'FAMILIAS',
       ...
       'NUMERO DE CONTRATO', 'CODIGO EMERGENCIA', 'FECHA DE ACTUALIZACION',
       'OBJETO', 'VALOR.35', 'VALOR.36', 'ATENDIDO.1', 'RESPONSABLE',
       'DESCRIPCIÓN', 'Unnamed: 120'],
      dtype='str', length=121)

In [ ]:
import math

cols = df_2017.columns.tolist()
total_cols = len(cols)
# Calculamos cuántas filas necesitamos (división hacia arriba)
filas = math.ceil(total_cols / 3) 

print(f"{'COLUMNA 1'.ljust(40)} | {'COLUMNA 2'.ljust(40)} | {'COLUMNA 3'}")
print("-" * 120)

for i in range(filas):
    col1 = cols[i] if i < total_cols else ""
    col2 = cols[i + filas] if (i + filas) < total_cols else ""
    col3 = cols[i + (filas * 2)] if (i + (filas * 2)) < total_cols else ""
    
    # Usamos ljust para que las tres columnas queden alineadas
    print(f"{col1.ljust(40)} | {col2.ljust(40)} | {col3}")

print("\n" + "-"*40)
print(f"Cantidad total de columnas: {total_cols}")

COLUMNA 1                                | COLUMNA 2                                | COLUMNA 3
------------------------------------------------------------------------------------------------------------------------
FECHA                                    | VALOR.1                                  | CANTIDAD.22
DEPARTAMENTO                             | CANTIDAD.2                               | VALOR.22
MUNICIPIO                                | VALOR.2                                  | CANTIDAD.23
EVENTO                                   | CANTIDAD.3                               | VALOR.23
Codificación según DIVIPOLA              | VALOR.3                                  | CANTIDAD.24
MUERTOS                                  | CANTIDAD.4                               | VALOR.24
HERIDOS                                  | VALOR.4                                  | CANTIDAD.25
DESAPA.                                  | CANTIDAD.5                               | VALOR.25
PERSONAS   

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "Codificación según DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_17 = extraer_columnas(df_2017, nom_col)

In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_17)

<class 'pandas.DataFrame'>
RangeIndex: 3363 entries, 0 to 3362
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   FECHA                        3338 non-null   object 
 1   Codificación según DIVIPOLA  3306 non-null   object 
 2   DEPARTAMENTO                 3339 non-null   str    
 3   MUNICIPIO                    3332 non-null   str    
 4   EVENTO                       3333 non-null   str    
 5   CODIGO EMERGENCIA            3333 non-null   float64
 6   PERSONAS                     1490 non-null   float64
 7   FAMILIAS                     1197 non-null   float64
 8   HECTAREAS                    1003 non-null   object 
 9   VIV.DESTRU.                  261 non-null    float64
 10  VIV.AVER.                    917 non-null    float64
 11  VIAS                         353 non-null    float64
 12  ACUED.                       90 non-null     float64
 13  OTROS                        

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_17['FECHA'] = pd.to_datetime(df_17['FECHA'], errors='coerce')
df_17 = extraer_año_mes(df_17,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_17.copy()
df_17 = filtrar_datos(df_17,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_17)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 645 entries, 4 to 3328
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        645 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  644 non-null    object        
 2   DEPARTAMENTO                 645 non-null    str           
 3   MUNICIPIO                    645 non-null    str           
 4   EVENTO                       645 non-null    str           
 5   CODIGO EMERGENCIA            645 non-null    float64       
 6   PERSONAS                     397 non-null    float64       
 7   FAMILIAS                     397 non-null    float64       
 8   HECTAREAS                    51 non-null     object        
 9   VIV.DESTRU.                  32 non-null     float64       
 10  VIV.AVER.                    291 non-null    float64       
 11  VIAS                         54 n

Unimos los resultados de deslizamiento e inundaciones

In [ ]:
#df_17 = unir_datasets(df_17,df_copy)

In [ ]:
# Borramos la fehca
#df_17 = borrar_columnas(df_17,"FECHA")

In [ ]:
info_dataset(df_17)

<class 'pandas.DataFrame'>
Index: 645 entries, 4 to 3328
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        645 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  644 non-null    object        
 2   DEPARTAMENTO                 645 non-null    str           
 3   MUNICIPIO                    645 non-null    str           
 4   EVENTO                       645 non-null    str           
 5   CODIGO EMERGENCIA            645 non-null    float64       
 6   PERSONAS                     397 non-null    float64       
 7   FAMILIAS                     397 non-null    float64       
 8   HECTAREAS                    51 non-null     object        
 9   VIV.DESTRU.                  32 non-null     float64       
 10  VIV.AVER.                    291 non-null    float64       
 11  VIAS                         54 non-null     float64       


##### Segundo análisis y preparación

In [ ]:
datos_nulos(df_17)

FECHA                            0
Codificación según DIVIPOLA      1
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                0
PERSONAS                       248
FAMILIAS                       248
HECTAREAS                      594
VIV.DESTRU.                    613
VIV.AVER.                      354
VIAS                           591
ACUED.                         621
OTROS                          634
COMENTARIOS                      0
VALOR.1                        645
FECHA_año                        0
FECHA_mes                        0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_17)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,Codificación según DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
68,2017-01-11,19809,CAUCA,TIMBIQUI,INUNDACION,67.0,14120.0,2824.0,2139,19.0,49.0,NaN,NaN,NaN,CDGRD CAUCA REPORTA AFECTACIONES EN EL MUNICIP...,NaN,2017.0,1.0
91,2017-01-14,73168,TOLIMA,CHAPARRAL,INUNDACION,74.0,135.0,27.0,NaN,NaN,25.0,NaN,NaN,NaN,CDGRD TOLIMA Y PONALSAR INFORMAN INUNDACIÓN P...,NaN,2017.0,1.0
100,2017-01-14,76364,VALLE DEL CAUCA,JAMUNDI,INUNDACION,103.0,200.0,40.0,300,NaN,NaN,NaN,NaN,17,CMGRD JAMUNDI REPORTA AFECTACION POR DESBORDAM...,NaN,2017.0,1.0
169,2017-01-20,52696,NARIÑO,SANTA BARBARA,INUNDACION,123.0,1201.0,421.0,560,NaN,210.0,NaN,NaN,NaN,"REPORTA EL CMGRD DE SANTA BARBARA DE ISCUANDE,...",NaN,2017.0,1.0
194,2017-01-21,41770,HUILA,SUAZA,INUNDACION,160.0,5.0,1.0,NaN,NaN,1.0,NaN,NaN,NaN,CDGRD DE HUILA INFORMA: M/PIO DE SUAZA: INUNDA...,NaN,2017.0,1.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 93 entries, 68 to 3174
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        93 non-null     datetime64[ns]
 1   Codificación según DIVIPOLA  93 non-null     object        
 2   DEPARTAMENTO                 93 non-null     str           
 3   MUNICIPIO                    93 non-null     str           
 4   EVENTO                       93 non-null     str           
 5   CODIGO EMERGENCIA            93 non-null     float64       
 6   PERSONAS                     61 non-null     float64       
 7   FAMILIAS                     63 non-null     float64       
 8   HECTAREAS                    31 non-null     object        
 9   VIV.DESTRU.                  7 non-null      float64       
 10  VIV.AVER.                    44 non-null     float64       
 11  VIAS                         5 non-null      float64       


In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        669 non-null    datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  609 non-null    object        
 2   DEPARTAMENTO                 669 non-null    str           
 3   MUNICIPIO                    669 non-null    str           
 4   EVENTO                       669 non-null    str           
 5   CODIGO EMERGENCIA            667 non-null    object        
 6   PERSONAS                     512 non-null    object        
 7   FAMILIAS                     556 non-null    object        
 8   HECTAREAS                    223 non-null    object        
 9   VIV.DESTRU.                  85 non-null     object        
 10  VIV.AVER.                    403 non-null    object        
 11  VIAS                         152 non-null    object     

##### Unión de los datasets

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"Codificación según DIVIPOLA","CODIFICACIÓN SEGUN DIVIPOLA")

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 762 entries, 0 to 761
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        762 non-null    datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  702 non-null    object        
 2   DEPARTAMENTO                 762 non-null    str           
 3   MUNICIPIO                    762 non-null    str           
 4   EVENTO                       762 non-null    str           
 5   CODIGO EMERGENCIA            760 non-null    object        
 6   PERSONAS                     573 non-null    object        
 7   FAMILIAS                     619 non-null    object        
 8   HECTAREAS                    254 non-null    object        
 9   VIV.DESTRU.                  92 non-null     object        
 10  VIV.AVER.                    447 non-null    object        
 11  VIAS                         157 non-null    object     

#### 2016

##### Exploración

In [ ]:
df_2016 = obtener_datos_para_modificados("../Data/UNGRD/EMERGENCIAS2016.xlsx")

In [ ]:
info_dataset(df_2016)

<class 'pandas.DataFrame'>
RangeIndex: 3879 entries, 0 to 3878
Columns: 120 entries, FECHA to DESCRIPCIÓN
dtypes: float64(103), object(6), str(11)
memory usage: 5.4+ MB


In [ ]:
df_2016.head()

,FECHA,DEPARTAMENTO,MUNICIPIO,EVENTO,Codificación según DIVIPOLA,MUERTOS,HERIDOS,DESAPA.,PERSONAS,FAMILIAS,...,NUMERO DE AFECTACION,NUMERO DE CONTRATO,CODIGO EMERGENCIA,FECHA DE ACTUALIZACION,OBJETO,VALOR.35,VALOR.36,ATENDIDO.1,RESPONSABLE,DESCRIPCIÓN
0,2016-01-01 00:00:00,ANTIOQUIA,RETIRO,INCENDIO FORESTAL,5607.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2016-01-01 00:00:00,NARIÑO,ANCUYA,INCENDIO ESTRUCTURAL,52036.0,NaN,NaN,NaN,5,1.0,...,NaN,NaN,2.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2016-01-01 00:00:00,CUNDINAMARCA,COTA,INCENDIO FORESTAL,25214.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2016-01-01 00:00:00,CUNDINAMARCA,EL COLEGIO,INCENDIO FORESTAL,25245.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,26.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2016-01-01 00:00:00,CUNDINAMARCA,BOJACA,INCENDIO FORESTAL,25099.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,27.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "Codificación según DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_16 = extraer_columnas(df_2016, nom_col)

In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_16)

<class 'pandas.DataFrame'>
RangeIndex: 3879 entries, 0 to 3878
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   FECHA                        3871 non-null   object 
 1   Codificación según DIVIPOLA  3866 non-null   float64
 2   DEPARTAMENTO                 3872 non-null   str    
 3   MUNICIPIO                    3866 non-null   str    
 4   EVENTO                       3866 non-null   str    
 5   CODIGO EMERGENCIA            3866 non-null   float64
 6   PERSONAS                     1427 non-null   object 
 7   FAMILIAS                     1175 non-null   float64
 8   HECTAREAS                    1574 non-null   float64
 9   VIV.DESTRU.                  247 non-null    float64
 10  VIV.AVER.                    891 non-null    float64
 11  VIAS                         254 non-null    float64
 12  ACUED.                       54 non-null     float64
 13  OTROS                        

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_16['FECHA'] = pd.to_datetime(df_16['FECHA'], errors='coerce')
df_16 = extraer_año_mes(df_16,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_16.copy()
df_16 = filtrar_datos(df_16,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_16)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 570 entries, 12 to 3861
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        570 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  570 non-null    float64       
 2   DEPARTAMENTO                 570 non-null    str           
 3   MUNICIPIO                    570 non-null    str           
 4   EVENTO                       570 non-null    str           
 5   CODIGO EMERGENCIA            570 non-null    float64       
 6   PERSONAS                     378 non-null    object        
 7   FAMILIAS                     377 non-null    float64       
 8   HECTAREAS                    45 non-null     float64       
 9   VIV.DESTRU.                  34 non-null     float64       
 10  VIV.AVER.                    285 non-null    float64       
 11  VIAS                         51 

Unión de los resultados

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"Codificación según DIVIPOLA","CODIFICACIÓN SEGUN DIVIPOLA")

In [ ]:
#df_16 = unir_datasets(df_16,df_copy)

In [ ]:
# Borramos la fehca
#df_16 = borrar_columnas(df_16,"FECHA")

In [ ]:
info_dataset(df_16)

<class 'pandas.DataFrame'>
Index: 570 entries, 12 to 3861
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        570 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  570 non-null    float64       
 2   DEPARTAMENTO                 570 non-null    str           
 3   MUNICIPIO                    570 non-null    str           
 4   EVENTO                       570 non-null    str           
 5   CODIGO EMERGENCIA            570 non-null    float64       
 6   PERSONAS                     378 non-null    object        
 7   FAMILIAS                     377 non-null    float64       
 8   HECTAREAS                    45 non-null     float64       
 9   VIV.DESTRU.                  34 non-null     float64       
 10  VIV.AVER.                    285 non-null    float64       
 11  VIAS                         51 non-null     float64       

##### Segundo análisis y preparación

In [ ]:
datos_nulos(df_16)

FECHA                            0
Codificación según DIVIPOLA      0
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                0
PERSONAS                       192
FAMILIAS                       193
HECTAREAS                      525
VIV.DESTRU.                    536
VIV.AVER.                      285
VIAS                           519
ACUED.                         554
OTROS                          559
COMENTARIOS                      0
VALOR.1                        570
FECHA_año                        0
FECHA_mes                        0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_16)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,Codificación según DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
1511,2016-03-31,19318.0,CAUCA,GUAPI,INUNDACION,1491.0,2760,460.0,690.0,NaN,NaN,NaN,NaN,NaN,CMGRD GUAPÍ INFORMA: INUNDACIÓN RIO SAN FRANCI...,NaN,2016.0,3.0
1538,2016-04-02,15572.0,BOYACA,PUERTO BOYACA,INUNDACION,1529.0,180,36.0,NaN,NaN,36.0,NaN,NaN,NaN,SOCORRO NACIONAL INFORMA: INUNDACIÓN EN PUERTO...,NaN,2016.0,4.0
1548,2016-04-03,76248.0,VALLE DEL CAUCA,EL CERRITO,INUNDACION,1534.0,35,6.0,NaN,NaN,6.0,NaN,NaN,NaN,CDGRD VALLE EN EL DÍA DE AYER EN EL MUNICIPIO ...,NaN,2016.0,4.0
1649,2016-04-12,27413.0,CHOCO,LLORO,INUNDACION,1720.0,445,89.0,1200.0,4.0,85.0,NaN,NaN,NaN,CDGRD INFORMA: SE PRESENTA DESBORDAMIENTO DEL ...,NaN,2016.0,4.0
1653,2016-04-13,52378.0,NARIÑO,LA CRUZ,INUNDACION,1649.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,CDGRD NARIÑO INFORMA QUE EN EL MUNICIPIO DE LA...,NaN,2016.0,4.0


##### Unión de los datasets

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"Codificación según DIVIPOLA","CODIFICACIÓN SEGUN DIVIPOLA")

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 833 entries, 0 to 832
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        833 non-null    datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  773 non-null    object        
 2   DEPARTAMENTO                 833 non-null    str           
 3   MUNICIPIO                    833 non-null    str           
 4   EVENTO                       833 non-null    str           
 5   CODIGO EMERGENCIA            831 non-null    object        
 6   PERSONAS                     632 non-null    object        
 7   FAMILIAS                     678 non-null    object        
 8   HECTAREAS                    280 non-null    object        
 9   VIV.DESTRU.                  103 non-null    object        
 10  VIV.AVER.                    481 non-null    object        
 11  VIAS                         166 non-null    object     

#### 2015

##### Exploración

In [ ]:
df_2015 = obtener_datos_para_modificados("../Data/UNGRD/EMERGENCIAS_2015.xlsx")

In [ ]:
info_dataset(df_2015)

<class 'pandas.DataFrame'>
RangeIndex: 3699 entries, 0 to 3698
Columns: 120 entries, FECHA to DESCRIPCIÓN
dtypes: datetime64[us](1), float64(103), object(5), str(11)
memory usage: 5.0+ MB


In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "Codificación según DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_15 = extraer_columnas(df_2015, nom_col)

In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_15)

<class 'pandas.DataFrame'>
RangeIndex: 3699 entries, 0 to 3698
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   FECHA                        3688 non-null   object 
 1   Codificación según DIVIPOLA  3683 non-null   float64
 2   DEPARTAMENTO                 3689 non-null   str    
 3   MUNICIPIO                    3683 non-null   str    
 4   EVENTO                       3683 non-null   str    
 5   CODIGO EMERGENCIA            3683 non-null   float64
 6   PERSONAS                     1197 non-null   float64
 7   FAMILIAS                     954 non-null    float64
 8   HECTAREAS                    1565 non-null   float64
 9   VIV.DESTRU.                  285 non-null    float64
 10  VIV.AVER.                    750 non-null    float64
 11  VIAS                         177 non-null    float64
 12  ACUED.                       59 non-null     float64
 13  OTROS                        

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_15['FECHA'] = pd.to_datetime(df_15['FECHA'], errors='coerce')
df_15 = extraer_año_mes(df_15,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_15.copy()
df_15 = filtrar_datos(df_15,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_15)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 256 entries, 237 to 3504
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        256 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  256 non-null    float64       
 2   DEPARTAMENTO                 256 non-null    str           
 3   MUNICIPIO                    256 non-null    str           
 4   EVENTO                       256 non-null    str           
 5   CODIGO EMERGENCIA            256 non-null    float64       
 6   PERSONAS                     180 non-null    float64       
 7   FAMILIAS                     180 non-null    float64       
 8   HECTAREAS                    30 non-null     float64       
 9   VIV.DESTRU.                  33 non-null     float64       
 10  VIV.AVER.                    161 non-null    float64       
 11  VIAS                         36

In [ ]:
#df_15 = unir_datasets(df_15,df_copy)

In [ ]:
# Borramos la fehca
#df_15 = borrar_columnas(df_15,"FECHA")

In [ ]:
info_dataset(df_15)

<class 'pandas.DataFrame'>
Index: 256 entries, 237 to 3504
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        256 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  256 non-null    float64       
 2   DEPARTAMENTO                 256 non-null    str           
 3   MUNICIPIO                    256 non-null    str           
 4   EVENTO                       256 non-null    str           
 5   CODIGO EMERGENCIA            256 non-null    float64       
 6   PERSONAS                     180 non-null    float64       
 7   FAMILIAS                     180 non-null    float64       
 8   HECTAREAS                    30 non-null     float64       
 9   VIV.DESTRU.                  33 non-null     float64       
 10  VIV.AVER.                    161 non-null    float64       
 11  VIAS                         36 non-null     float64      

##### Segundo análisis y preparation

In [ ]:
datos_nulos(df_15)

FECHA                            0
Codificación según DIVIPOLA      0
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                0
PERSONAS                        76
FAMILIAS                        76
HECTAREAS                      226
VIV.DESTRU.                    223
VIV.AVER.                       95
VIAS                           220
ACUED.                         235
OTROS                          253
COMENTARIOS                      0
VALOR.1                        256
FECHA_año                        0
FECHA_mes                        0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_15)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,Codificación según DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
237,2015-01-16,27025.0,CHOCO,ALTO BAUDO,INUNDACION,241.0,2000.0,400.0,NaN,NaN,400.0,NaN,NaN,NaN,"CDGRD DEL CHOCO Y CMGRD DEL ALTO BAUDO, INFORM...",NaN,2015.0,1.0
247,2015-01-18,27099.0,CHOCO,BOJAYA,INUNDACION,252.0,5160.0,860.0,NaN,NaN,860.0,NaN,NaN,NaN,CMGRD BOJAYA DR. JOSE MENA INFORMA UNA INUNDAC...,NaN,2015.0,1.0
373,2015-02-05,19701.0,CAUCA,SANTA ROSA,INUNDACION,364.0,535.0,107.0,NaN,NaN,107.0,NaN,NaN,NaN,"5 DE FEBRERO DEL AÑO EN CURSO, EN EL CORREGIMI...",NaN,2015.0,2.0
421,2015-02-12,86001.0,PUTUMAYO,MOCOA,INUNDACION,414.0,25.0,5.0,NaN,NaN,5.0,NaN,NaN,NaN,"CDGRD DEL PUTUMAYO, INFORMA, SE PRESENTÓ, CREC...",NaN,2015.0,2.0
960,2015-03-20,86320.0,PUTUMAYO,ORITO,INUNDACION,935.0,125.0,25.0,NaN,NaN,25.0,NaN,NaN,NaN,CDGRD PUTUMAYO INFORMA CRECIENTE EN ORITO RIO...,NaN,2015.0,3.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 47 entries, 237 to 3450
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        47 non-null     datetime64[ns]
 1   Codificación según DIVIPOLA  47 non-null     float64       
 2   DEPARTAMENTO                 47 non-null     str           
 3   MUNICIPIO                    47 non-null     str           
 4   EVENTO                       47 non-null     str           
 5   CODIGO EMERGENCIA            47 non-null     float64       
 6   PERSONAS                     44 non-null     float64       
 7   FAMILIAS                     45 non-null     float64       
 8   HECTAREAS                    16 non-null     float64       
 9   VIV.DESTRU.                  16 non-null     float64       
 10  VIV.AVER.                    41 non-null     float64       
 11  VIAS                         14 non-null     float64       

##### Unión de los datasets

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"Codificación según DIVIPOLA","CODIFICACIÓN SEGUN DIVIPOLA")

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 880 entries, 0 to 879
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        880 non-null    datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  820 non-null    object        
 2   DEPARTAMENTO                 880 non-null    str           
 3   MUNICIPIO                    880 non-null    str           
 4   EVENTO                       880 non-null    str           
 5   CODIGO EMERGENCIA            878 non-null    object        
 6   PERSONAS                     676 non-null    object        
 7   FAMILIAS                     723 non-null    object        
 8   HECTAREAS                    296 non-null    object        
 9   VIV.DESTRU.                  119 non-null    object        
 10  VIV.AVER.                    522 non-null    object        
 11  VIAS                         180 non-null    object     

#### Consideración a obtener estos datos

Se considera obtener ya que hasta el año 2015 no hay suficientes datos para tener una base solida a analizar.

### 2014

##### Exploración

In [ ]:
df_2014 = obtener_datos_para_modificados("../Data/UNGRD/EMERGENCIAS2014.xlsx")

In [ ]:
info_dataset(df_2014)

<class 'pandas.DataFrame'>
RangeIndex: 3693 entries, 0 to 3692
Columns: 120 entries, FECHA to DESCRIPCIÓN
dtypes: datetime64[us](1), float64(96), object(9), str(14)
memory usage: 5.0+ MB


In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "Codificación según DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "CODIGO EMERGENCIA", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_14 = extraer_columnas(df_2014, nom_col)

In [ ]:
# Vemos el resultado de la extración de las columnas
info_dataset(df_14)

<class 'pandas.DataFrame'>
RangeIndex: 3693 entries, 0 to 3692
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   FECHA                        3684 non-null   object 
 1   Codificación según DIVIPOLA  3679 non-null   float64
 2   DEPARTAMENTO                 3685 non-null   str    
 3   MUNICIPIO                    3679 non-null   str    
 4   EVENTO                       3679 non-null   str    
 5   CODIGO EMERGENCIA            3679 non-null   float64
 6   PERSONAS                     1548 non-null   object 
 7   FAMILIAS                     1212 non-null   object 
 8   HECTAREAS                    1231 non-null   float64
 9   VIV.DESTRU.                  357 non-null    object 
 10  VIV.AVER.                    936 non-null    float64
 11  VIAS                         255 non-null    float64
 12  ACUED.                       77 non-null     float64
 13  OTROS                        

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_14['FECHA'] = pd.to_datetime(df_14['FECHA'], errors='coerce')
df_14 = extraer_año_mes(df_14,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_14.copy()
df_14 = filtrar_datos(df_14,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_14)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 463 entries, 27 to 3678
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        463 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  463 non-null    float64       
 2   DEPARTAMENTO                 463 non-null    str           
 3   MUNICIPIO                    463 non-null    str           
 4   EVENTO                       463 non-null    str           
 5   CODIGO EMERGENCIA            463 non-null    float64       
 6   PERSONAS                     305 non-null    object        
 7   FAMILIAS                     302 non-null    object        
 8   HECTAREAS                    36 non-null     float64       
 9   VIV.DESTRU.                  41 non-null     object        
 10  VIV.AVER.                    281 non-null    float64       
 11  VIAS                         47 

Unimos los resultados

In [ ]:
#df_14 = unir_datasets(df_14,df_copy)

In [ ]:
# Borramos la fehca
#df_14 = borrar_columnas(df_14,"FECHA")

In [ ]:
info_dataset(df_14)

<class 'pandas.DataFrame'>
Index: 463 entries, 27 to 3678
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        463 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  463 non-null    float64       
 2   DEPARTAMENTO                 463 non-null    str           
 3   MUNICIPIO                    463 non-null    str           
 4   EVENTO                       463 non-null    str           
 5   CODIGO EMERGENCIA            463 non-null    float64       
 6   PERSONAS                     305 non-null    object        
 7   FAMILIAS                     302 non-null    object        
 8   HECTAREAS                    36 non-null     float64       
 9   VIV.DESTRU.                  41 non-null     object        
 10  VIV.AVER.                    281 non-null    float64       
 11  VIAS                         47 non-null     float64       

##### Segundo análisis y perparación de los datos

In [ ]:
print("FILAS/COLUMNAS",df_14.shape)
datos_nulos(df_14)

FECHA                            0
Codificación según DIVIPOLA      0
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
CODIGO EMERGENCIA                0
PERSONAS                       158
FAMILIAS                       161
HECTAREAS                      427
VIV.DESTRU.                    422
VIV.AVER.                      182
VIAS                           416
ACUED.                         431
OTROS                          447
COMENTARIOS                      0
VALOR.1                        463
FECHA_año                        0
FECHA_mes                        0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_14)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,Codificación según DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
27,2014-01-05,25214.0,CUNDINAMARCA,COTA,INUNDACION,30.0,25,5,NaN,NaN,5.0,NaN,NaN,NaN,"D.C.C., INFORMA, SIENDO LAS 16:00 HORAS DE HOY...",NaN,2014.0,1.0
69,2014-01-09,19622.0,CAUCA,ROSAS,INUNDACION,72.0,843,338,266.0,NaN,338.0,NaN,1.0,NaN,AFECTADA ZONA URBANA Y RURAL. PERDIDA DE CULTIVOS,NaN,2014.0,1.0
71,2014-01-09,19780.0,CAUCA,SUAREZ,INUNDACION,1564.0,985,197,NaN,NaN,197.0,NaN,NaN,NaN,SE REPORTA AFECTACIÓN POR LLUVIAS PRESENTADAS ...,NaN,2014.0,1.0
755,2014-03-12,41885.0,HUILA,YAGUARA,INUNDACION,751.0,5,1,14.0,NaN,1.0,NaN,NaN,NaN,"CDGRD DEL HUILA, INFORMA, MUNICIPIO YAGUARA EV...",NaN,2014.0,3.0
769,2014-03-13,41801.0,HUILA,TERUEL,INUNDACION,760.0,20,4,NaN,1,3.0,NaN,NaN,NaN,INFORMAN DEL CDGRD DEL HUILA QUE SE PRESENTO U...,NaN,2014.0,3.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 70 entries, 27 to 3576
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        70 non-null     datetime64[ns]
 1   Codificación según DIVIPOLA  70 non-null     float64       
 2   DEPARTAMENTO                 70 non-null     str           
 3   MUNICIPIO                    70 non-null     str           
 4   EVENTO                       70 non-null     str           
 5   CODIGO EMERGENCIA            70 non-null     float64       
 6   PERSONAS                     54 non-null     object        
 7   FAMILIAS                     53 non-null     object        
 8   HECTAREAS                    26 non-null     float64       
 9   VIV.DESTRU.                  9 non-null      object        
 10  VIV.AVER.                    49 non-null     float64       
 11  VIAS                         13 non-null     float64       


##### Unión de dataset

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"Codificación según DIVIPOLA","CODIFICACIÓN SEGUN DIVIPOLA")

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        950 non-null    datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  890 non-null    object        
 2   DEPARTAMENTO                 950 non-null    str           
 3   MUNICIPIO                    950 non-null    str           
 4   EVENTO                       950 non-null    str           
 5   CODIGO EMERGENCIA            948 non-null    object        
 6   PERSONAS                     730 non-null    object        
 7   FAMILIAS                     776 non-null    object        
 8   HECTAREAS                    322 non-null    object        
 9   VIV.DESTRU.                  128 non-null    object        
 10  VIV.AVER.                    571 non-null    object        
 11  VIAS                         193 non-null    object     

### 2013

##### Exploración

In [ ]:
df_2013 = obtener_datos_alterno("../Data/UNGRD/EMERGENCIAS2013.xlsx")

In [ ]:
info_dataset(df_2013)

<class 'pandas.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Columns: 128 entries, FECHA to Unnamed: 127
dtypes: float64(110), object(6), str(12)
memory usage: 7.3+ MB


In [ ]:
df_2013.columns

Index(['FECHA', 'DEPARTAMENTO', 'MUNICIPIO', 'EVENTO',
       'Codificación según DIVIPOLA', 'MUERTOS', 'HERIDOS', 'DESAPA.',
       'PERSONAS', 'FAMILIAS',
       ...
       'Unnamed: 118', 'Unnamed: 119', 'Unnamed: 120', 'Unnamed: 121',
       'Unnamed: 122', 'Unnamed: 123', 'Unnamed: 124', 'Unnamed: 125',
       'Unnamed: 126', 'Unnamed: 127'],
      dtype='str', length=128)

In [ ]:
import math

cols = df_2013.columns.tolist()
total_cols = len(cols)
# Calculamos cuántas filas necesitamos (división hacia arriba)
filas = math.ceil(total_cols / 3) 

print(f"{'COLUMNA 1'.ljust(40)} | {'COLUMNA 2'.ljust(40)} | {'COLUMNA 3'}")
print("-" * 120)

for i in range(filas):
    col1 = cols[i] if i < total_cols else ""
    col2 = cols[i + filas] if (i + filas) < total_cols else ""
    col3 = cols[i + (filas * 2)] if (i + (filas * 2)) < total_cols else ""
    
    # Usamos ljust para que las tres columnas queden alineadas
    print(f"{col1.ljust(40)} | {col2.ljust(40)} | {col3}")

print("\n" + "-"*40)
print(f"Cantidad total de columnas: {total_cols}")

COLUMNA 1                                | COLUMNA 2                                | COLUMNA 3
------------------------------------------------------------------------------------------------------------------------
FECHA                                    | VALOR.2                                  | CANTIDAD.24
DEPARTAMENTO                             | CANTIDAD.3                               | VALOR.24
MUNICIPIO                                | VALOR.3                                  | CANTIDAD.25
EVENTO                                   | CANTIDAD.4                               | VALOR.25
Codificación según DIVIPOLA              | VALOR.4                                  | CANTIDAD.26
MUERTOS                                  | CANTIDAD.5                               | VALOR.26
HERIDOS                                  | VALOR.5                                  | CANTIDAD.27
DESAPA.                                  | CANTIDAD.6                               | VALOR.27
PERSONAS   

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "Codificación según DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "NUMERO DE RADICADO", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_13 = extraer_columnas(df_2013, nom_col)

In [ ]:
info_dataset(df_13)

<class 'pandas.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   FECHA                        4024 non-null   object 
 1   Codificación según DIVIPOLA  4019 non-null   float64
 2   DEPARTAMENTO                 4025 non-null   str    
 3   MUNICIPIO                    4019 non-null   str    
 4   EVENTO                       4019 non-null   str    
 5   NUMERO DE RADICADO           1 non-null      float64
 6   PERSONAS                     1763 non-null   float64
 7   FAMILIAS                     1633 non-null   float64
 8   HECTAREAS                    1126 non-null   float64
 9   VIV.DESTRU.                  423 non-null    float64
 10  VIV.AVER.                    1364 non-null   float64
 11  VIAS                         270 non-null    float64
 12  ACUED.                       76 non-null     float64
 13  OTROS                        

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_13['FECHA'] = pd.to_datetime(df_13['FECHA'], errors='coerce')
df_13 = extraer_año_mes(df_13,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_13.copy()
df_13 = filtrar_datos(df_13,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_13)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 668 entries, 226 to 3981
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        668 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  668 non-null    float64       
 2   DEPARTAMENTO                 668 non-null    str           
 3   MUNICIPIO                    668 non-null    str           
 4   EVENTO                       668 non-null    str           
 5   NUMERO DE RADICADO           1 non-null      float64       
 6   PERSONAS                     441 non-null    float64       
 7   FAMILIAS                     438 non-null    float64       
 8   HECTAREAS                    42 non-null     float64       
 9   VIV.DESTRU.                  67 non-null     float64       
 10  VIV.AVER.                    402 non-null    float64       
 11  VIAS                         51

In [ ]:
#df_13 = unir_datasets(df_13,df_copy)

In [ ]:
# Borramos la fehca
#df_13 = borrar_columnas(df_13,"FECHA")

In [ ]:
info_dataset(df_13)

<class 'pandas.DataFrame'>
Index: 668 entries, 226 to 3981
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        668 non-null    datetime64[ns]
 1   Codificación según DIVIPOLA  668 non-null    float64       
 2   DEPARTAMENTO                 668 non-null    str           
 3   MUNICIPIO                    668 non-null    str           
 4   EVENTO                       668 non-null    str           
 5   NUMERO DE RADICADO           1 non-null      float64       
 6   PERSONAS                     441 non-null    float64       
 7   FAMILIAS                     438 non-null    float64       
 8   HECTAREAS                    42 non-null     float64       
 9   VIV.DESTRU.                  67 non-null     float64       
 10  VIV.AVER.                    402 non-null    float64       
 11  VIAS                         51 non-null     float64      

##### Segundo análisis y preparación de datos

In [ ]:
datos_nulos(df_13)

FECHA                            0
Codificación según DIVIPOLA      0
DEPARTAMENTO                     0
MUNICIPIO                        0
EVENTO                           0
NUMERO DE RADICADO             667
PERSONAS                       227
FAMILIAS                       230
HECTAREAS                      626
VIV.DESTRU.                    601
VIV.AVER.                      266
VIAS                           617
ACUED.                         640
OTROS                          648
COMENTARIOS                     16
VALOR.1                        668
FECHA_año                        0
FECHA_mes                        0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_13)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,Codificación según DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,NUMERO DE RADICADO,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
664,2013-02-09,41016.0,HUILA,AIPE,INUNDACION,NaN,75.0,15.0,0.5,NaN,14.0,NaN,NaN,90,REPORTA INUNDACION A CAUSA DEL DESBORDAMIENTO ...,NaN,2013.0,2.0
983,2013-03-01,27430.0,CHOCO,MEDIO BAUDO,INUNDACION,NaN,620.0,92.0,NaN,NaN,92.0,NaN,NaN,NaN,"CDGRD DEL CHOCO, INFORMA, DESBORDAMIENTO DEL R...",NaN,2013.0,3.0
1139,2013-03-15,5790.0,ANTIOQUIA,TARAZA,INUNDACION,NaN,125.0,25.0,20.0,NaN,25.0,NaN,NaN,NaN,"D.C.C, INFORMA, DESDE LA NOCHE ANTERIOR DEL 15...",NaN,2013.0,3.0
1144,2013-03-15,27025.0,CHOCO,ALTO BAUDO,INUNDACION,NaN,1000.0,200.0,NaN,NaN,NaN,NaN,NaN,NaN,"CDGRD DEL CHOCO, INFORMA, DESBORDAMIENTO DEL R...",NaN,2013.0,3.0
1145,2013-03-15,27430.0,CHOCO,MEDIO BAUDO,INUNDACION,NaN,16185.0,3237.0,1939.0,482.0,259.0,NaN,NaN,NaN,"CMGRD DEL CHOCO, INFORMA, DESBORDAMIENTO DEL R...",NaN,2013.0,3.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 67 entries, 664 to 3981
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        67 non-null     datetime64[ns]
 1   Codificación según DIVIPOLA  67 non-null     float64       
 2   DEPARTAMENTO                 67 non-null     str           
 3   MUNICIPIO                    67 non-null     str           
 4   EVENTO                       67 non-null     str           
 5   NUMERO DE RADICADO           0 non-null      float64       
 6   PERSONAS                     60 non-null     float64       
 7   FAMILIAS                     60 non-null     float64       
 8   HECTAREAS                    30 non-null     float64       
 9   VIV.DESTRU.                  14 non-null     float64       
 10  VIV.AVER.                    55 non-null     float64       
 11  VIAS                         9 non-null      float64       

##### Unión de los datasets

In [ ]:
#df_filtrado = cambiar_nombre_columna(df_filtrado,"DEPTO","DEPARTAMENTO")
df_filtrado = cambiar_nombre_columna(df_filtrado,"Codificación según DIVIPOLA","CODIFICACIÓN SEGUN DIVIPOLA")
df_filtrado = cambiar_nombre_columna(df_filtrado,"NUMERO DE RADICADO", "CODIGO EMERGENCIA")

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 1017 entries, 0 to 1016
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        1017 non-null   datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  957 non-null    object        
 2   DEPARTAMENTO                 1017 non-null   str           
 3   MUNICIPIO                    1017 non-null   str           
 4   EVENTO                       1017 non-null   str           
 5   CODIGO EMERGENCIA            948 non-null    object        
 6   PERSONAS                     790 non-null    object        
 7   FAMILIAS                     836 non-null    object        
 8   HECTAREAS                    352 non-null    object        
 9   VIV.DESTRU.                  142 non-null    object        
 10  VIV.AVER.                    626 non-null    object        
 11  VIAS                         202 non-null    object   

### 2012

##### Exploración

In [ ]:
df_2012 = obtener_datos_alterno("../Data/UNGRD/EMERGENCIAS2012.xlsx")

In [ ]:
info_dataset(df_2012)

<class 'pandas.DataFrame'>
RangeIndex: 4621 entries, 0 to 4620
Columns: 111 entries, FECHA to Unnamed: 110
dtypes: float64(93), object(11), str(7)
memory usage: 4.1+ MB


In [ ]:
df_2012.columns

Index(['FECHA', 'DEPARTAMENTO', 'MUNICIPIO', 'EVENTO',
       'Codificación según DIVIPOLA', 'Unnamed: 5', 'MUERTOS', 'HERIDOS',
       'DESAPA.', 'PERSONAS',
       ...
       'VALOR.30', 'CANTIDAD.31', 'VALOR.31', 'CANTIDAD.32', 'VALOR.32',
       'CANTIDAD.33', 'VALOR.33', 'LINEA NO.', 'Unnamed: 109', 'Unnamed: 110'],
      dtype='str', length=111)

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "Codificación según DIVIPOLA","DEPARTAMENTO", "MUNICIPIO", "EVENTO", "NUMERO DE RADICADO", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_12 = extraer_columnas(df_2012, nom_col)

In [ ]:
info_dataset(df_12)

<class 'pandas.DataFrame'>
RangeIndex: 4621 entries, 0 to 4620
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   FECHA                        4045 non-null   object 
 1   Codificación según DIVIPOLA  4044 non-null   float64
 2   DEPARTAMENTO                 4044 non-null   str    
 3   MUNICIPIO                    4044 non-null   str    
 4   EVENTO                       4044 non-null   str    
 5   NUMERO DE RADICADO           0 non-null      float64
 6   PERSONAS                     1619 non-null   float64
 7   FAMILIAS                     1594 non-null   object 
 8   HECTAREAS                    1113 non-null   object 
 9   VIV.DESTRU.                  399 non-null    float64
 10  VIV.AVER.                    1330 non-null   float64
 11  VIAS                         446 non-null    float64
 12  ACUED.                       162 non-null    object 
 13  OTROS                        

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_12['FECHA'] = pd.to_datetime(df_12['FECHA'], errors='coerce')
df_12 = extraer_año_mes(df_12,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_12.copy()
df_12 = filtrar_datos(df_12,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_12)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 1265 entries, 16 to 4018
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        1265 non-null   datetime64[us]
 1   Codificación según DIVIPOLA  1265 non-null   float64       
 2   DEPARTAMENTO                 1265 non-null   str           
 3   MUNICIPIO                    1265 non-null   str           
 4   EVENTO                       1265 non-null   str           
 5   NUMERO DE RADICADO           0 non-null      float64       
 6   PERSONAS                     559 non-null    float64       
 7   FAMILIAS                     554 non-null    object        
 8   HECTAREAS                    59 non-null     object        
 9   VIV.DESTRU.                  84 non-null     float64       
 10  VIV.AVER.                    462 non-null    float64       
 11  VIAS                         13

In [ ]:
#df_12 = unir_datasets(df_12,df_copy)

In [ ]:
# Borramos la fehca
#df_12 = borrar_columnas(df_12,"FECHA")

In [ ]:
info_dataset(df_12)

<class 'pandas.DataFrame'>
Index: 1265 entries, 16 to 4018
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        1265 non-null   datetime64[us]
 1   Codificación según DIVIPOLA  1265 non-null   float64       
 2   DEPARTAMENTO                 1265 non-null   str           
 3   MUNICIPIO                    1265 non-null   str           
 4   EVENTO                       1265 non-null   str           
 5   NUMERO DE RADICADO           0 non-null      float64       
 6   PERSONAS                     559 non-null    float64       
 7   FAMILIAS                     554 non-null    object        
 8   HECTAREAS                    59 non-null     object        
 9   VIV.DESTRU.                  84 non-null     float64       
 10  VIV.AVER.                    462 non-null    float64       
 11  VIAS                         130 non-null    float64      

##### Segundo análisis y preparacion

In [ ]:
datos_nulos(df_12)

FECHA                             0
Codificación según DIVIPOLA       0
DEPARTAMENTO                      0
MUNICIPIO                         0
EVENTO                            0
NUMERO DE RADICADO             1265
PERSONAS                        706
FAMILIAS                        711
HECTAREAS                      1206
VIV.DESTRU.                    1181
VIV.AVER.                       803
VIAS                           1135
ACUED.                         1201
OTROS                          1259
COMENTARIOS                      87
VALOR.1                        1265
FECHA_año                         0
FECHA_mes                         0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_12)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,Codificación según DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,NUMERO DE RADICADO,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
33,2012-01-05,52696.0,NARIÑO,SANTA BARBARA,INUNDACION,NaN,1662.0,363,373,4.0,359.0,NaN,NaN,NaN,SE PRESENTO DESBORDAMIENTO DEL RIO ISCUANDE EN...,NaN,2012.0,1.0
88,2012-01-13,15676.0,BOYACA,SAN MIGUEL DE SEMA,INUNDACION,NaN,3164.0,791,6043,NaN,NaN,NaN,NaN,NaN,"SECTORES AFECTADOS SABANECA, QUINTOQUE, PEÑA ...",NaN,2012.0,1.0
140,2012-01-18,19318.0,CAUCA,GUAPI,INUNDACION,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DESBORDAMIENTO DEL RIO GUAPI-AFECTACION A CUL...,NaN,2012.0,1.0
194,2012-01-23,41483.0,HUILA,NATAGA,INUNDACION,NaN,NaN,NaN,4.5,NaN,NaN,NaN,NaN,NaN,CRECIENTE DE LA QUEBRADA FUENTE DE PIEDRA -AFE...,NaN,2012.0,1.0
239,2012-01-25,41006.0,HUILA,ACEVEDO,INUNDACION,NaN,155.0,31,11,2.0,18.0,3.0,NaN,NaN,"DAÑOS PRESENTADOS EN LAS VEREDA PUEBLO VIEJO, ...",NaN,2012.0,1.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 68 entries, 33 to 3812
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        68 non-null     datetime64[us]
 1   Codificación según DIVIPOLA  68 non-null     float64       
 2   DEPARTAMENTO                 68 non-null     str           
 3   MUNICIPIO                    68 non-null     str           
 4   EVENTO                       68 non-null     str           
 5   NUMERO DE RADICADO           0 non-null      float64       
 6   PERSONAS                     54 non-null     float64       
 7   FAMILIAS                     54 non-null     object        
 8   HECTAREAS                    37 non-null     object        
 9   VIV.DESTRU.                  15 non-null     float64       
 10  VIV.AVER.                    48 non-null     float64       
 11  VIAS                         27 non-null     float64       


##### Unión de los datasets

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"DEPTO","DEPARTAMENTO")
df_filtrado = cambiar_nombre_columna(df_filtrado,"Codificación según DIVIPOLA","CODIFICACIÓN SEGUN DIVIPOLA")
df_filtrado = cambiar_nombre_columna(df_filtrado,"NUMERO DE RADICADO", "CODIGO EMERGENCIA")

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 1085 entries, 0 to 1084
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        1085 non-null   datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  1025 non-null   object        
 2   DEPARTAMENTO                 1085 non-null   str           
 3   MUNICIPIO                    1085 non-null   str           
 4   EVENTO                       1085 non-null   str           
 5   CODIGO EMERGENCIA            948 non-null    object        
 6   PERSONAS                     844 non-null    object        
 7   FAMILIAS                     890 non-null    object        
 8   HECTAREAS                    389 non-null    object        
 9   VIV.DESTRU.                  157 non-null    object        
 10  VIV.AVER.                    674 non-null    object        
 11  VIAS                         229 non-null    object   

### 2011

##### Exploración

In [ ]:
df_2011 = obtener_datos_alterno("../Data/UNGRD/EMERGENCIAS2011.xlsx")

In [ ]:
info_dataset(df_2011)

<class 'pandas.DataFrame'>
RangeIndex: 3805 entries, 0 to 3804
Columns: 111 entries, FECHA to Unnamed: 110
dtypes: float64(86), object(17), str(8)
memory usage: 3.4+ MB


In [ ]:
df_2011.columns

Index(['FECHA', 'DEPTO', 'MUNICIPIO', 'EVENTO', 'Codificación según DIVIPOLA',
       'Unnamed: 5', 'MUERTOS', 'HERIDOS', 'DESAPA.', 'PERSONAS',
       ...
       'VALOR.30', 'CANTIDAD.31', 'VALOR.31', 'CANTIDAD.32', 'VALOR.32',
       'CANTIDAD.33', 'VALOR.33', 'LINEA NO.', 'Unnamed: 109', 'Unnamed: 110'],
      dtype='str', length=111)

In [ ]:
# Columnas seleccionadas
nom_col = ["FECHA", "Codificación según DIVIPOLA","DEPTO", "MUNICIPIO", "EVENTO", "NUMERO DE RADICADO", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_11 = extraer_columnas(df_2011, nom_col)

In [ ]:
info_dataset(df_11)

<class 'pandas.DataFrame'>
RangeIndex: 3805 entries, 0 to 3804
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   FECHA                        3201 non-null   object 
 1   Codificación según DIVIPOLA  3198 non-null   float64
 2   DEPTO                        3200 non-null   str    
 3   MUNICIPIO                    3200 non-null   str    
 4   EVENTO                       3201 non-null   str    
 5   NUMERO DE RADICADO           0 non-null      float64
 6   PERSONAS                     2369 non-null   object 
 7   FAMILIAS                     2364 non-null   float64
 8   HECTAREAS                    241 non-null    object 
 9   VIV.DESTRU.                  558 non-null    float64
 10  VIV.AVER.                    1836 non-null   float64
 11  VIAS                         440 non-null    object 
 12  ACUED.                       127 non-null    float64
 13  OTROS                        

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_11['FECHA'] = pd.to_datetime(df_11['FECHA'], errors='coerce')
df_11 = extraer_año_mes(df_11,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_11.copy()
df_11 = filtrar_datos(df_11,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_11)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 1684 entries, 3 to 3199
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        1684 non-null   datetime64[us]
 1   Codificación según DIVIPOLA  1684 non-null   float64       
 2   DEPTO                        1684 non-null   str           
 3   MUNICIPIO                    1684 non-null   str           
 4   EVENTO                       1684 non-null   str           
 5   NUMERO DE RADICADO           0 non-null      float64       
 6   PERSONAS                     1220 non-null   object        
 7   FAMILIAS                     1219 non-null   float64       
 8   HECTAREAS                    68 non-null     object        
 9   VIV.DESTRU.                  192 non-null    float64       
 10  VIV.AVER.                    1028 non-null   float64       
 11  VIAS                         160

Se unen los resultados de inundaciones y deslizamientos

In [ ]:
#df_11 = unir_datasets(df_11,df_copy)

In [ ]:
# Borramos la fehca
#df_11 = borrar_columnas(df_11,"FECHA")

In [ ]:
info_dataset(df_11)

<class 'pandas.DataFrame'>
Index: 1684 entries, 3 to 3199
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        1684 non-null   datetime64[us]
 1   Codificación según DIVIPOLA  1684 non-null   float64       
 2   DEPTO                        1684 non-null   str           
 3   MUNICIPIO                    1684 non-null   str           
 4   EVENTO                       1684 non-null   str           
 5   NUMERO DE RADICADO           0 non-null      float64       
 6   PERSONAS                     1220 non-null   object        
 7   FAMILIAS                     1219 non-null   float64       
 8   HECTAREAS                    68 non-null     object        
 9   VIV.DESTRU.                  192 non-null    float64       
 10  VIV.AVER.                    1028 non-null   float64       
 11  VIAS                         160 non-null    object        

###### Segundo análisis y praparación de los datos

In [ ]:
datos_nulos(df_11)

FECHA                             0
Codificación según DIVIPOLA       0
DEPTO                             0
MUNICIPIO                         0
EVENTO                            0
NUMERO DE RADICADO             1684
PERSONAS                        464
FAMILIAS                        465
HECTAREAS                      1616
VIV.DESTRU.                    1492
VIV.AVER.                       656
VIAS                           1524
ACUED.                         1631
OTROS                          1654
COMENTARIOS                      89
VALOR.1                        1684
FECHA_año                         0
FECHA_mes                         0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_11)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,Codificación según DIVIPOLA,DEPTO,MUNICIPIO,EVENTO,NUMERO DE RADICADO,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
270,2011-03-20,27025.0,CHOCO,ALTO BAUDO,INUNDACION,NaN,5012,1129.0,NaN,NaN,1129.0,NaN,NaN,PERDIDA DE CULTIVOS. Y ANIMALES DOMESTICOS.,"COMUNIDADES AFECTADAS PIE DE PATO, PUERTO ECEV...",NaN,2011.0,3.0
348,2011-04-01,73067.0,TOLIMA,ATACO,INUNDACION,NaN,5624,1457.0,NaN,60.0,405.0,12,6.0,PERDIDA DE CULTIVOS,DEBIDO A LA TEMPORADA INVERNAL SE PRESENTARON ...,NaN,2011.0,4.0
415,2011-04-09,20770.0,CESAR,SAN MARTIN,INUNDACION,NaN,250,50.0,NaN,NaN,50.0,NaN,NaN,PERDIDA EN CULTIVOS Y ANIMALES DOMESTICOS.,CRECIENTE CAÑO TORCOROMA.CORREGIMIENTO DE MINA...,NaN,2011.0,4.0
478,2011-04-12,73504.0,TOLIMA,ORTEGA,INUNDACION,NaN,680,151.0,NaN,NaN,151.0,NaN,NaN,NaN,"AFECTACION Y PERDIDAS EN CULTIVOS, REPORTO CRE...",NaN,2011.0,4.0
592,2011-04-17,41016.0,HUILA,AIPE,INUNDACION,NaN,160,32.0,NaN,NaN,NaN,NaN,NaN,NaN,"VEREDAS: PATA, PIEDRA PINTADA, LA ISLA, EL RIN...",NaN,2011.0,4.0


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 89 entries, 270 to 3133
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        89 non-null     datetime64[us]
 1   Codificación según DIVIPOLA  89 non-null     float64       
 2   DEPTO                        89 non-null     str           
 3   MUNICIPIO                    89 non-null     str           
 4   EVENTO                       89 non-null     str           
 5   NUMERO DE RADICADO           0 non-null      float64       
 6   PERSONAS                     78 non-null     object        
 7   FAMILIAS                     79 non-null     float64       
 8   HECTAREAS                    39 non-null     object        
 9   VIV.DESTRU.                  22 non-null     float64       
 10  VIV.AVER.                    69 non-null     float64       
 11  VIAS                         20 non-null     object        

##### Unión del dataset

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"DEPTO","DEPARTAMENTO")
df_filtrado = cambiar_nombre_columna(df_filtrado,"Codificación según DIVIPOLA","CODIFICACIÓN SEGUN DIVIPOLA")
df_filtrado = cambiar_nombre_columna(df_filtrado,"NUMERO DE RADICADO", "CODIGO EMERGENCIA")

In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 1174 entries, 0 to 1173
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        1174 non-null   datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  1114 non-null   object        
 2   DEPARTAMENTO                 1174 non-null   str           
 3   MUNICIPIO                    1174 non-null   str           
 4   EVENTO                       1174 non-null   str           
 5   CODIGO EMERGENCIA            948 non-null    object        
 6   PERSONAS                     922 non-null    object        
 7   FAMILIAS                     969 non-null    object        
 8   HECTAREAS                    428 non-null    object        
 9   VIV.DESTRU.                  179 non-null    object        
 10  VIV.AVER.                    743 non-null    object        
 11  VIAS                         249 non-null    object   

### 2010

##### Exploración

In [ ]:
df_2010 = obtener_datos_alterno("../Data/UNGRD/EMERGENCIAS_2010.xlsx")

In [ ]:
info_dataset(df_2010)

<class 'pandas.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Columns: 151 entries, FECHA to Criterio_Llave
dtypes: datetime64[us](3), float64(131), object(5), str(12)
memory usage: 1.0+ MB


In [ ]:
df_2010.columns

Index(['FECHA', 'DEPTO', 'MUNICIPIO', 'EVENTO', 'Codificación según DIVIPOLA',
       'Unnamed: 5', 'MUERTOS', 'HERIDOS', 'DESAPA.', 'PERSONAS',
       ...
       'Unnamed: 141', 'Unnamed: 142', 'Unnamed: 143', 'Llave', 'Unnamed: 145',
       'Cod_Dpto', 'Nombre Dpto', 'Cod_Mpio', 'Nombre Mpio', 'Criterio_Llave'],
      dtype='str', length=151)

In [ ]:
# Columnas seleccionadas
# Columnas seleccionadas
nom_col = ["FECHA", "Codificación según DIVIPOLA","DEPTO", "MUNICIPIO", "EVENTO", "NUMERO DE RADICADO", "PERSONAS",
           "FAMILIAS", "HECTAREAS", "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.", "OTROS", "COMENTARIOS", "VALOR.1"]
# Se extrae
df_10 = extraer_columnas(df_2010, nom_col)

In [ ]:
info_dataset(df_10)

<class 'pandas.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        731 non-null    datetime64[us]
 1   Codificación según DIVIPOLA  0 non-null      float64       
 2   DEPTO                        731 non-null    str           
 3   MUNICIPIO                    731 non-null    str           
 4   EVENTO                       731 non-null    str           
 5   NUMERO DE RADICADO           233 non-null    object        
 6   PERSONAS                     448 non-null    float64       
 7   FAMILIAS                     448 non-null    float64       
 8   HECTAREAS                    31 non-null     float64       
 9   VIV.DESTRU.                  98 non-null     float64       
 10  VIV.AVER.                    310 non-null    float64       
 11  VIAS                         53 non-null     float64    

##### Preparación

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df_10['FECHA'] = pd.to_datetime(df_10['FECHA'], errors='coerce')
df_10 = extraer_año_mes(df_10,"FECHA")

In [ ]:
# Filtramos los datos a inundaciones y/o deslizamientos
df_copy = df_10.copy()
df_10 = filtrar_datos(df_10,"EVENTO","INUNDACION")
df_copy = filtrar_datos(df_copy,"EVENTO","DESLIZAMIENTO")

In [ ]:
# Mostramos la información del dataset en cuanto a los filtros
print("Dataframe con inundaciones")
info_dataset(df_10)
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("-----------------------------------------------------------------------------------------------------------------")
print("Dataframe con deslizamientos")
info_dataset(df_copy)

Dataframe con inundaciones
<class 'pandas.DataFrame'>
Index: 217 entries, 3 to 726
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        217 non-null    datetime64[us]
 1   Codificación según DIVIPOLA  0 non-null      float64       
 2   DEPTO                        217 non-null    str           
 3   MUNICIPIO                    217 non-null    str           
 4   EVENTO                       217 non-null    str           
 5   NUMERO DE RADICADO           63 non-null     object        
 6   PERSONAS                     184 non-null    float64       
 7   FAMILIAS                     184 non-null    float64       
 8   HECTAREAS                    15 non-null     float64       
 9   VIV.DESTRU.                  33 non-null     float64       
 10  VIV.AVER.                    150 non-null    float64       
 11  VIAS                         24 no

Unión de los resultados

In [ ]:
# Borramos la fehca
#df_10 = borrar_columnas(df_10,"FECHA")

In [ ]:
info_dataset(df_10)

<class 'pandas.DataFrame'>
Index: 217 entries, 3 to 726
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        217 non-null    datetime64[us]
 1   Codificación según DIVIPOLA  0 non-null      float64       
 2   DEPTO                        217 non-null    str           
 3   MUNICIPIO                    217 non-null    str           
 4   EVENTO                       217 non-null    str           
 5   NUMERO DE RADICADO           63 non-null     object        
 6   PERSONAS                     184 non-null    float64       
 7   FAMILIAS                     184 non-null    float64       
 8   HECTAREAS                    15 non-null     float64       
 9   VIV.DESTRU.                  33 non-null     float64       
 10  VIV.AVER.                    150 non-null    float64       
 11  VIAS                         24 non-null     float64       
 

##### Segundo análisis y preparación del dataset

In [ ]:
datos_nulos(df_10)

FECHA                            0
Codificación según DIVIPOLA    217
DEPTO                            0
MUNICIPIO                        0
EVENTO                           0
NUMERO DE RADICADO             154
PERSONAS                        33
FAMILIAS                        33
HECTAREAS                      202
VIV.DESTRU.                    184
VIV.AVER.                       67
VIAS                           193
ACUED.                         206
OTROS                          192
COMENTARIOS                      2
VALOR.1                        217
FECHA_año                        0
FECHA_mes                        0
dtype: int64


In [ ]:
# Filtrar el DataFrame
df_filtrado = filtrar_por_palabras_clave(df_10)

# Mostrar el DataFrame filtrado
df_filtrado.head()

,FECHA,Codificación según DIVIPOLA,DEPTO,MUNICIPIO,EVENTO,NUMERO DE RADICADO,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,VIV.AVER.,VIAS,ACUED.,OTROS,COMENTARIOS,VALOR.1,FECHA_año,FECHA_mes
220,2010-02-24,NaN,CHOCO,MEDIO ATRATO,INUNDACION,95577,2293.0,461.0,NaN,3.0,NaN,NaN,NaN,PERDIDA DE CULTIVOS Y ANIMALES DOMESTICOS,DESBORDAMIENTO RIO BUEY. AFECTADAS COMUNIDADES...,NaN,2010,2
262,2010-03-10,NaN,CAQUETA,FLORENCIA,INUNDACION,96283,3500.0,700.0,NaN,6.0,694.0,3.0,NaN,AFECTADO CLUB DEL COMERCIO. PERDIDA DE CULTIVO...,"DESBORDAMIENT0 QUEBRADA LA YUCA, VEREDAS SAN ...",NaN,2010,3
296,2010-03-29,NaN,ANTIOQUIA,NECOCLI,INUNDACION,96686,1750.0,350.0,NaN,NaN,350.0,NaN,NaN,PERDIDA DE CULTIVOS,"SECTORES DE SEMANA SANTA, ALGODÓN ARRIBA, ALGO...",NaN,2010,3
357,2010-04-09,NaN,META,SAN JUAN DE ARAMA,INUNDACION,96554,123.0,30.0,180.0,NaN,30.0,2.0,1.0,PERDIDA DE CULTIVOS Y ESPECIES MENORES,"VEREDAS MIRAFLORES, LA DESPENSA, LA BALASTRERA...",NaN,2010,4
374,2010-04-11,NaN,META,PUERTO LOPEZ,INUNDACION,NaN,300.0,60.0,15.0,NaN,60.0,NaN,NaN,"CULTIVOS DE PLATANO, YUCA YY AVES",DESBORDAMIENTO DEL CAÑO PARRADO VEREDA PACHAQU...,NaN,2010,4


In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 18 entries, 220 to 699
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        18 non-null     datetime64[us]
 1   Codificación según DIVIPOLA  0 non-null      float64       
 2   DEPTO                        18 non-null     str           
 3   MUNICIPIO                    18 non-null     str           
 4   EVENTO                       18 non-null     str           
 5   NUMERO DE RADICADO           10 non-null     object        
 6   PERSONAS                     17 non-null     float64       
 7   FAMILIAS                     17 non-null     float64       
 8   HECTAREAS                    8 non-null      float64       
 9   VIV.DESTRU.                  5 non-null      float64       
 10  VIV.AVER.                    12 non-null     float64       
 11  VIAS                         7 non-null      float64       


##### Unión de los datasets

In [ ]:
df_filtrado = cambiar_nombre_columna(df_filtrado,"DEPTO","DEPARTAMENTO")
df_filtrado = cambiar_nombre_columna(df_filtrado,"Codificación según DIVIPOLA","CODIFICACIÓN SEGUN DIVIPOLA")
df_filtrado = cambiar_nombre_columna(df_filtrado,"NUMERO DE RADICADO", "CODIGO EMERGENCIA")

In [ ]:
info_dataset(df_filtrado)

<class 'pandas.DataFrame'>
Index: 18 entries, 220 to 699
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        18 non-null     datetime64[us]
 1   CODIFICACIÓN SEGUN DIVIPOLA  0 non-null      float64       
 2   DEPARTAMENTO                 18 non-null     str           
 3   MUNICIPIO                    18 non-null     str           
 4   EVENTO                       18 non-null     str           
 5   CODIGO EMERGENCIA            10 non-null     object        
 6   PERSONAS                     17 non-null     float64       
 7   FAMILIAS                     17 non-null     float64       
 8   HECTAREAS                    8 non-null      float64       
 9   VIV.DESTRU.                  5 non-null      float64       
 10  VIV.AVER.                    12 non-null     float64       
 11  VIAS                         7 non-null      float64       


In [ ]:
df_definitivo = unir_datasets(df_definitivo,df_filtrado)

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   FECHA                        1192 non-null   datetime64[ns]
 1   CODIFICACIÓN SEGUN DIVIPOLA  1114 non-null   object        
 2   DEPARTAMENTO                 1192 non-null   str           
 3   MUNICIPIO                    1192 non-null   str           
 4   EVENTO                       1192 non-null   str           
 5   CODIGO EMERGENCIA            958 non-null    object        
 6   PERSONAS                     939 non-null    object        
 7   FAMILIAS                     986 non-null    object        
 8   HECTAREAS                    436 non-null    object        
 9   VIV.DESTRU.                  184 non-null    object        
 10  VIV.AVER.                    755 non-null    object        
 11  VIAS                         256 non-null    object   

In [ ]:
porcentaje_nulos(df_definitivo)

FECHA                          0.000000
CODIFICACIÓN SEGUN DIVIPOLA    0.065436
DEPARTAMENTO                   0.000000
MUNICIPIO                      0.000000
EVENTO                         0.000000
CODIGO EMERGENCIA              0.196309
PERSONAS                       0.212248
FAMILIAS                       0.172819
HECTAREAS                      0.634228
VIV.DESTRU.                    0.845638
VIV.AVER.                      0.366611
VIAS                           0.785235
ACUED.                         0.873322
OTROS                          0.762584
COMENTARIOS                    0.000839
VALOR.1                        0.632550
FECHA_año                      0.057886
FECHA_mes                      0.057886
dtype: float64


El porcentaje de nulos superan el 10%, esto afirma el problema al intentar explorar maneras de transformación

## Resultados

El dataframe y nuevo dataset a obtener es la extracción de variables que coinciden a través del tiempo, sin embargo esta sujeto a cambios en dado caso que se quiera obtener más información, el problema que impide la consideración de más variable (columnas) es la cantidad de nulos que esto poseen, harían atropellado el tema de una limpieza y reparación de los mismo ya que al considerar formas estos poseen más del 80% de datos pérdidos.

Por otro lado, la información está cruzada, es decir, los datos con las categorias de inundaciones y deslizamientos tienen sub-afectaciones en cultivos, viviendas, estructuras, etc., filtrar aún más el número de personas, familias y valor (costo) por solo cultivos es complicado y filtar aún más el dataset sería inoportuno por los problemas ya exhibidos anteriomente.

(Queda bajo consideración, pero abierto a cambios)

Se cambio el nombre a algunas variables por practicidad

In [ ]:
df_definitivo = cambiar_nombre_columna(df_definitivo, "CODIFICACIÓN SEGUN DIVIPOLA", "DIVIPOLA")

In [ ]:
info_dataset(df_definitivo)

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   FECHA              1192 non-null   datetime64[ns]
 1   DIVIPOLA           1114 non-null   object        
 2   DEPARTAMENTO       1192 non-null   str           
 3   MUNICIPIO          1192 non-null   str           
 4   EVENTO             1192 non-null   str           
 5   CODIGO EMERGENCIA  958 non-null    object        
 6   PERSONAS           939 non-null    object        
 7   FAMILIAS           986 non-null    object        
 8   HECTAREAS          436 non-null    object        
 9   VIV.DESTRU.        184 non-null    object        
 10  VIV.AVER.          755 non-null    object        
 11  VIAS               256 non-null    object        
 12  ACUED.             151 non-null    object        
 13  OTROS              283 non-null    object        
 14  COMENTARIOS        

In [ ]:
# Eliminamos variables para liberar espacio
del df_10, df_11, df_12, df_13, df_14, df_15, df_16, df_17, df_18, df_19, df_20, df_21, df_22, df_23, df_24
del df_2010, df_2011, df_2012, df_2013, df_2014, df_2015, df_2016, df_2017, df_2018, df_2019, df_2020, df_2021, df_2022, df_2023

## Preparación de resultados

### Librerias

In [ ]:
import unicodedata
import gc

gc.collect()

965

### Funciones de preparación de resultados

In [ ]:
def Arreglo_fecha(dat):
    dat['FECHA'] = pd.to_datetime(dat['FECHA'], errors='coerce')
    dat['year'] = dat['FECHA'].dt.year
    dat['month'] = dat['FECHA'].dt.month
    dat['ym'] = dat['FECHA'].dt.to_period('M') # Formato 'YYYY-MM'
    return dat

In [ ]:
def Banderas_davipola(dat):
    # Creamos la bandera de nulos antes de imputar
    dat['missing_divipola_flag'] = dat['DIVIPOLA'].isna().astype(int)

    # Extraemos solo los números, quitamos decimales (.0) y rellenamos con ceros a la izquierda
    dat['DIVIPOLA'] = (dat['DIVIPOLA']
                            .astype(str)
                            .str.extract(r'(\d+)', expand=False)
                            .str.zfill(5))
    # Volvemos nulos los que eran originalmente NaN (que se volvieron 'nan' o '00nan')
    dat.loc[dat['missing_divipola_flag'] == 1, 'DIVIPOLA'] = np.nan
    return dat

In [ ]:
# Funcion para quitar tildes y mayúsculas de DEPARTAMENTO y MUNICIPIO
def clean_location_text(text):
    if pd.isna(text):
        return text
    # Convertir a mayúsculas
    text = str(text).upper().strip()
    # Quitar tildes (normalización NFD elimina diacríticos)
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    return text

### Preparación

#### Correciones y limpieza base

In [ ]:
# Borramos la fehca
df_definitivo = borrar_columnas(df_definitivo,"FECHA_año")
df_definitivo = borrar_columnas(df_definitivo,"FECHA_mes")

In [ ]:
# Copiamos el dataframe ya procesado para no alterar negativamente los resultados
df = df_definitivo.copy() # SObre df se harán los cambios de preparación de resultados

In [ ]:
# Extraemos el año y mes a la variable fecha, esto nos permite diferenciar el tiempo que ocurrio
df = Arreglo_fecha(df)

In [ ]:
# Se crea la variable bandera de valores perdidos en DAVIPOLA
df = Banderas_davipola(df)

In [ ]:
# Se transforman las variables str u object a por defecto a numericas
cols_numericas = ['PERSONAS', 'FAMILIAS', 'HECTAREAS', 'VIV.DESTRU.', 'VIV.AVER.', 'VIAS', 'ACUED.', 'VALOR.1']
for col in cols_numericas:
    # errors='coerce' convierte textos raros o espacios a NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Limpiamoslas palabras en dado caso que tengas tildes y mayusculas
df['DEPARTAMENTO'] = df['DEPARTAMENTO'].apply(clean_location_text)
df['MUNICIPIO'] = df['MUNICIPIO'].apply(clean_location_text)

In [ ]:
info_dataset(df)

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   FECHA                  1192 non-null   datetime64[ns]
 1   DIVIPOLA               1105 non-null   str           
 2   DEPARTAMENTO           1192 non-null   str           
 3   MUNICIPIO              1192 non-null   str           
 4   EVENTO                 1192 non-null   str           
 5   CODIGO EMERGENCIA      958 non-null    object        
 6   PERSONAS               939 non-null    float64       
 7   FAMILIAS               986 non-null    float64       
 8   HECTAREAS              434 non-null    float64       
 9   VIV.DESTRU.            184 non-null    float64       
 10  VIV.AVER.              755 non-null    float64       
 11  VIAS                   256 non-null    float64       
 12  ACUED.                 151 non-null    float64       
 13  OTROS         

#### Variables derivadas de la base de la UNGRD

In [ ]:
# Limpiar textos de OTROS y COMENTARIOS asegurando que todo sea tipo 'string'
texto_combinado = df['OTROS'].fillna('').astype(str) + " " + df['COMENTARIOS'].fillna('').astype(str)

# Normalizamos el texto combinado (minúsculas y sin tildes) para no fallar en la búsqueda
texto_combinado_limpio = texto_combinado.apply(lambda x: ''.join(c for c in unicodedata.normalize('NFD', str(x).lower()) if unicodedata.category(c) != 'Mn'))

# Definir palabras clave (sin tildes por la normalización previa)
keywords_cultivos = [
    r'perdida de cultivos?', r'danos en cultivos?', r'cultivos? afectados?', 
    r'cultivos? de pancoger', r'pancoger', r'maiz', r'platano', r'yuca', r'arroz', r'cafe'
]
patron_cultivos = '|'.join(keywords_cultivos)
# flood_crop_text_flag clave para resumir la base donde la afectación aparece narrada
df['flood_crop_text_flag'] = texto_combinado_limpio.str.contains(patron_cultivos, regex=True).astype(int)

In [ ]:
# flood_crop_quantified_flag y hectareas_reported_flag
df['hectareas_reported_flag'] = df['HECTAREAS'].notna().astype(int)
df['flood_crop_quantified_flag'] = (df['HECTAREAS'] > 0).astype(int)

In [ ]:
# flood_crop_any_flag
df['flood_crop_any_flag'] = df[['flood_crop_text_flag', 'flood_crop_quantified_flag']].max(axis=1)

In [ ]:
# event_id_clean
# Limpiamos espacios y caracteres raros del código de emergencia
df['event_id_clean'] = df['CODIGO EMERGENCIA'].astype(str).str.strip().str.upper()

#### Variables secundarias

Estas nuevas variables se devriban como opcionales en caso de llegarlas a necesitar

In [ ]:
# Rellenamos con 0 solo para la suma, así evitamos que NaN + Valor = NaN
df['housing_damage_proxy'] = df['VIV.DESTRU.'].fillna(0) + df['VIV.AVER.'].fillna(0)
df['infra_damage_proxy'] = df['VIAS'].fillna(0) + df['ACUED.'].fillna(0)

# Para support_fngrd_value, simplemente renombramos o copiamos
df['support_fngrd_value'] = df['VALOR.1']

In [ ]:
info_dataset(df)

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   FECHA                       1192 non-null   datetime64[ns]
 1   DIVIPOLA                    1105 non-null   str           
 2   DEPARTAMENTO                1192 non-null   str           
 3   MUNICIPIO                   1192 non-null   str           
 4   EVENTO                      1192 non-null   str           
 5   CODIGO EMERGENCIA           958 non-null    object        
 6   PERSONAS                    939 non-null    float64       
 7   FAMILIAS                    986 non-null    float64       
 8   HECTAREAS                   434 non-null    float64       
 9   VIV.DESTRU.                 184 non-null    float64       
 10  VIV.AVER.                   755 non-null    float64       
 11  VIAS                        256 non-null    float64       
 12  ACU

In [ ]:
df.columns

Index(['FECHA', 'DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO', 'EVENTO',
       'CODIGO EMERGENCIA', 'PERSONAS', 'FAMILIAS', 'HECTAREAS', 'VIV.DESTRU.',
       'VIV.AVER.', 'VIAS', 'ACUED.', 'OTROS', 'COMENTARIOS', 'VALOR.1',
       'year', 'month', 'ym', 'missing_divipola_flag', 'flood_crop_text_flag',
       'hectareas_reported_flag', 'flood_crop_quantified_flag',
       'flood_crop_any_flag', 'event_id_clean', 'housing_damage_proxy',
       'infra_damage_proxy', 'support_fngrd_value'],
      dtype='str')

Una vez listo y preparado el dataset, pasamos a limipiar el codigo divipola

In [ ]:
# Cargar y preparar el diccionario DIVIPOLA oficial
df_divipola = pd.read_excel('../Data/DIVIPOLA_Municipios.xlsx', skiprows=10)

In [ ]:
df_divipola.columns

Index(['Código', 'Nombre', ' Código ', ' Nombre ',
       'Municipio\nIsla\nÁrea no municipalizada', 'Longitud', 'Latitud',
       'Nota'],
      dtype='str')

In [ ]:
# Renombramos las columnas para que sean fáciles de manejar (según la estructura que compartiste)
df_divipola = df_divipola.rename(columns={
    'Nombre': 'DEP_NOMBRE', 
    ' Código ': 'DIVIPOLA_MUNI', # El código de 5 dígitos suele ser la segunda columna de código
    ' Nombre ': 'MUN_NOMBRE'
})

In [ ]:
info_dataset(df_divipola)

<class 'pandas.DataFrame'>
RangeIndex: 1132 entries, 0 to 1131
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Código                                 1127 non-null   str    
 1   DEP_NOMBRE                             1122 non-null   str    
 2   DIVIPOLA_MUNI                          1122 non-null   float64
 3   MUN_NOMBRE                             1122 non-null   str    
 4   Municipio
Isla
Área no municipalizada  1122 non-null   str    
 5   Longitud                               1122 non-null   float64
 6   Latitud                                1122 non-null   float64
 7   Nota                                   4 non-null      str    
dtypes: float64(3), str(5)
memory usage: 104.8 KB


In [ ]:
df_divipola['DEP_NOMBRE'] = df_divipola['DEP_NOMBRE'].apply(clean_location_text)
df_divipola['MUN_NOMBRE'] = df_divipola['MUN_NOMBRE'].apply(clean_location_text)

In [ ]:
# Extraemos solo los 5 dígitos, quitamos decimales y rellenamos con 0 a la izquierda
df_divipola['DIVIPOLA_MUNI'] = df_divipola['DIVIPOLA_MUNI'].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(5)

In [ ]:
# Hacemos el cruce para traer el código correcto
df = df.merge(
    df_divipola[['DEP_NOMBRE', 'MUN_NOMBRE', 'DIVIPOLA_MUNI']],
    left_on=['DEPARTAMENTO', 'MUNICIPIO'],
    right_on=['DEP_NOMBRE', 'MUN_NOMBRE'],
    how='left'
)

In [ ]:
df.head()

,FECHA,DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,...,hectareas_reported_flag,flood_crop_quantified_flag,flood_crop_any_flag,event_id_clean,housing_damage_proxy,infra_damage_proxy,support_fngrd_value,DEP_NOMBRE,MUN_NOMBRE,DIVIPOLA_MUNI
0,2024-02-07,19809,CAUCA,TIMBIQUI,INUNDACION,2468.0,2611.0,629.0,NaN,NaN,...,0,0,1,2468.0,19.0,0.0,0.0,CAUCA,TIMBIQUI,19809
1,2024-03-13,27430,CHOCO,MEDIO BAUDO,INUNDACION,4342.0,12500.0,4050.0,760.0,7.0,...,1,1,1,4342.0,7.0,0.0,0.0,CHOCO,MEDIO BAUDO,27430
2,2024-03-13,27025,CHOCO,ALTO BAUDO,INUNDACION,4352.0,15000.0,7500.0,23000.0,35.0,...,1,1,1,4352.0,7535.0,10.0,0.0,CHOCO,ALTO BAUDO,27025
3,2024-04-01,19809,CAUCA,TIMBIQUI,INUNDACION,5207.0,3391.0,980.0,633.0,4.0,...,1,1,1,5207.0,125.0,2.0,0.0,CAUCA,TIMBIQUI,19809
4,2024-04-05,05147,ANTIOQUIA,CAREPA,INUNDACION,5304.0,7500.0,1875.0,270.0,NaN,...,1,1,1,5304.0,0.0,0.0,0.0,ANTIOQUIA,CAREPA,05147


In [ ]:
# Reemplazamos la columna DAVIPOLA original con la oficial
df['DIVIPOLA'] = df['DIVIPOLA_MUNI']
df = df.drop(columns=['DEP_NOMBRE', 'MUN_NOMBRE', 'DIVIPOLA_MUNI'])

print(f"Nulos restantes en DAVIPOLA tras el cruce: {df['DIVIPOLA'].isna().sum()}")

Nulos restantes en DAVIPOLA tras el cruce: 28


In [ ]:
info_dataset(df)

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   FECHA                       1192 non-null   datetime64[ns]
 1   DIVIPOLA                    1164 non-null   str           
 2   DEPARTAMENTO                1192 non-null   str           
 3   MUNICIPIO                   1192 non-null   str           
 4   EVENTO                      1192 non-null   str           
 5   CODIGO EMERGENCIA           958 non-null    object        
 6   PERSONAS                    939 non-null    float64       
 7   FAMILIAS                    986 non-null    float64       
 8   HECTAREAS                   434 non-null    float64       
 9   VIV.DESTRU.                 184 non-null    float64       
 10  VIV.AVER.                   755 non-null    float64       
 11  VIAS                        256 non-null    float64       
 12  ACU

# Objención de imagenes para el modelo U-net

In [ ]:
RUTA_XLSX = "../Data_preparada/datos_inundaciones_v2.xlsx"
RUTA_SALIDA = "eventos_inundacion_lote.csv"

# Sentinel-1 útil desde 2014-10-03
FECHA_MIN_S1 = pd.Timestamp("2014-10-03")

df = pd.read_excel(RUTA_XLSX)

# Limpieza básica
df["FECHA"] = pd.to_datetime(df["FECHA"], errors="coerce")
df["DIVIPOLA"] = (
    df["DIVIPOLA"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.zfill(5)
)

df["MUNICIPIO"] = df["MUNICIPIO"].astype(str).str.upper().str.strip()

# Filtrar fechas compatibles con Sentinel-1
df = df[df["FECHA"] >= FECHA_MIN_S1].copy()

# Quitar filas sin codigo o fecha
df = df[df["DIVIPOLA"].notna() & df["FECHA"].notna()].copy()

# Quedarse con columnas clave
df = df[["FECHA", "DIVIPOLA", "MUNICIPIO"]].copy()

# Deduplicar por municipio + fecha
df = df.drop_duplicates(subset=["DIVIPOLA", "FECHA"]).copy()

# Nombre de evento para exportacion
df["nombre_evento"] = (
    df["MUNICIPIO"].str.replace(" ", "_", regex=False)
    + "_"
    + df["DIVIPOLA"]
    + "_"
    + df["FECHA"].dt.strftime("%Y_%m_%d")
)

# Umbrales generales
df["umbral_agua_despues_vv"] = -18.0
df["umbral_cambio_vv"] = -1.5
df["umbral_pendiente"] = 5.0
df["umbral_agua_permanente"] = 50.0
df["buffer_metros"] = 2000

# Ventanas temporales generales
df["dias_antes_inicio"] = 60
df["dias_antes_fin"] = 10
df["dias_despues_inicio"] = 0
df["dias_despues_fin"] = 30

# Renombrar
df = df.rename(columns={
    "FECHA": "fecha_evento",
    "DIVIPOLA": "codigo_municipio",
    "MUNICIPIO": "municipio"
})

df.to_csv(RUTA_SALIDA, index=False, encoding="utf-8-sig")

print("Archivo generado:", RUTA_SALIDA)
print("Registros:", len(df))
print(df.head())

Archivo generado: eventos_inundacion_lote.csv
Registros: 869
  fecha_evento codigo_municipio    municipio                 nombre_evento  \
0   2024-02-07            19809     TIMBIQUI     TIMBIQUI_19809_2024_02_07   
1   2024-03-13            27430  MEDIO BAUDO  MEDIO_BAUDO_27430_2024_03_13   
2   2024-03-13            27025   ALTO BAUDO   ALTO_BAUDO_27025_2024_03_13   
3   2024-04-01            19809     TIMBIQUI     TIMBIQUI_19809_2024_04_01   
4   2024-04-05            05147       CAREPA       CAREPA_05147_2024_04_05   

   umbral_agua_despues_vv  umbral_cambio_vv  umbral_pendiente  \
0                   -18.0              -1.5               5.0   
1                   -18.0              -1.5               5.0   
2                   -18.0              -1.5               5.0   
3                   -18.0              -1.5               5.0   
4                   -18.0              -1.5               5.0   

   umbral_agua_permanente  buffer_metros  dias_antes_inicio  dias_antes_fin  \


In [ ]:
import ee
import pandas as pd
import time

# =========================================================
# 1) CONFIGURACION GENERAL
# =========================================================
RUTA_XLSX = "../Data_preparada/datos_inundaciones_v2.xlsx"   # cambia si hace falta

ASSET_BASE = "projects/precipitacion-485912/assets/"
ASSET_MUNICIPIOS = ASSET_BASE + "igac_mun"
ASSET_FRONTERA = ASSET_BASE + "Frontera_Agricola_Jun2025"
ASSET_APTITUD_MAIZ = ASSET_BASE + "Aptitud_Maiz_CalidoS1_Dic2019"
ASSET_APTITUD_ARROZ = ASSET_BASE + "aptitud_arrozsecano_dic2019"

CARPETA_DRIVE = "GEE_MASCARA_INUNDACION_AGRICOLA"
#dfgdfg #Ya cambié, pero lo colo para recordar
# Lotes
LOTE_INICIO = 900
LOTE_FIN = 950 # cambia por corrida: 0-50, 50-100, etc.

# Resolucion recomendada para ahorrar espacio
ESCALA_EXPORTACION = 30

# Ventanas temporales
DIAS_ANTES_INICIO = 60
DIAS_ANTES_FIN = 10
DIAS_DESPUES_INICIO = 0
DIAS_DESPUES_FIN = 30

# Umbrales generales
UMBRAL_AGUA_DESPUES_VV = -18.0
UMBRAL_CAMBIO_VV = -1.5
UMBRAL_PENDIENTE = 5.0
UMBRAL_AGUA_PERMANENTE = 50.0

# Recorte util
BUFFER_MUNICIPIO_METROS = 2000
BUFFER_BBOX_UTIL_METROS = 500

# Filtrar ruido muy pequeño
AREA_MINIMA_HA = 1.0

# Sentinel-1 util desde octubre de 2014
FECHA_MIN_S1 = pd.Timestamp("2014-10-03")

# =========================================================
# 2) INICIALIZAR EARTH ENGINE
# =========================================================
import ee
ee.Authenticate() 
# Luego inicializas
ee.Initialize(project='precipitacion-485912')


# =========================================================
# 3) CARGAR ASSETS
# =========================================================
municipios = ee.FeatureCollection(ASSET_MUNICIPIOS)
frontera = ee.FeatureCollection(ASSET_FRONTERA)
aptitud_maiz = ee.FeatureCollection(ASSET_APTITUD_MAIZ)
aptitud_arroz = ee.FeatureCollection(ASSET_APTITUD_ARROZ)

# =========================================================
# 4) CARGAR Y PREPARAR EVENTOS DESDE EL EXCEL
# =========================================================
df = pd.read_excel(RUTA_XLSX)

# Ajusta estos nombres si en tu archivo vienen distintos
df["FECHA"] = pd.to_datetime(df["FECHA"], errors="coerce")
df["DIVIPOLA"] = (
    df["DIVIPOLA"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.zfill(5)
)
df["MUNICIPIO"] = df["MUNICIPIO"].astype(str).str.upper().str.strip()

# Filtrar fechas utiles para Sentinel-1
df = df[df["FECHA"] >= FECHA_MIN_S1].copy()

# Quitar incompletos
df = df[df["FECHA"].notna() & df["DIVIPOLA"].notna()].copy()

# Deduplicar por municipio + fecha
df = df.drop_duplicates(subset=["DIVIPOLA", "FECHA"]).copy()

# Nombre amigable de salida
df["nombre_evento"] = (
    df["MUNICIPIO"].str.replace(" ", "_", regex=False)
    + "_"
    + df["DIVIPOLA"]
    + "_"
    + df["FECHA"].dt.strftime("%Y_%m_%d")
)

# Renombrar para trabajar mejor
df = df.rename(columns={
    "DIVIPOLA": "codigo_municipio",
    "MUNICIPIO": "municipio",
    "FECHA": "fecha_evento"
})

# Subconjunto por lote
eventos = df.iloc[LOTE_INICIO:LOTE_FIN].copy()

print("Eventos en lote:", len(eventos))
print(eventos.head())

# =========================================================
# 5) FUNCIONES
# =========================================================
def construir_mascara_agricola(aoi):
    union_agricola = (
        frontera.filterBounds(aoi)
        .merge(aptitud_maiz.filterBounds(aoi))
        .merge(aptitud_arroz.filterBounds(aoi))
    )

    mascara = (
        ee.Image()
        .byte()
        .paint(featureCollection=union_agricola, color=1)
        .rename("mascara_agricola")
        .unmask(0)
    )
    return mascara


def construir_mascara_inundacion_agricola(codigo_municipio, fecha_evento):
    # -----------------------------------------------------
    # Municipio
    # -----------------------------------------------------
    municipio = municipios.filter(ee.Filter.eq("MPCOD", str(codigo_municipio)))
    n_municipio = municipio.size().getInfo()

    if n_municipio == 0:
        return {"estado": "municipio_no_encontrado"}

    aoi = municipio.geometry().bounds(BUFFER_MUNICIPIO_METROS)

    # -----------------------------------------------------
    # Fechas
    # -----------------------------------------------------
    f_evento = ee.Date(str(fecha_evento)[:10])

    antes_inicio = f_evento.advance(-DIAS_ANTES_INICIO, "day")
    antes_fin = f_evento.advance(-DIAS_ANTES_FIN, "day")
    despues_inicio = f_evento.advance(DIAS_DESPUES_INICIO, "day")
    despues_fin = f_evento.advance(DIAS_DESPUES_FIN, "day")

    # -----------------------------------------------------
    # Sentinel-1 solo VV
    # -----------------------------------------------------
    s1 = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(antes_inicio, despues_fin)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .select("VV")
    )

    s1_antes = s1.filterDate(antes_inicio, antes_fin)
    s1_despues = s1.filterDate(despues_inicio, despues_fin)

    n_antes = s1_antes.size().getInfo()
    n_despues = s1_despues.size().getInfo()

    if n_antes == 0 or n_despues == 0:
        return {
            "estado": "sin_escenas_s1",
            "escenas_antes": n_antes,
            "escenas_despues": n_despues
        }

    antes = ee.Image(s1_antes.median()).rename("s1_antes_vv")
    despues = ee.Image(s1_despues.median()).rename("s1_despues_vv")

    # Suavizado
    antes = antes.focal_median(radius=30, units="meters")
    despues = despues.focal_median(radius=30, units="meters")

    delta_vv = despues.subtract(antes).rename("delta_vv")

    # -----------------------------------------------------
    # Regla de inundacion
    # -----------------------------------------------------
    agua_despues = despues.lt(UMBRAL_AGUA_DESPUES_VV)
    cambio_fuerte = delta_vv.lt(UMBRAL_CAMBIO_VV)

    dem = ee.Image("USGS/SRTMGL1_003")
    pendiente = ee.Terrain.slope(dem)
    mascara_pendiente = pendiente.lt(UMBRAL_PENDIENTE)

    gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    agua_permanente = gsw.select("occurrence").gte(UMBRAL_AGUA_PERMANENTE)

    mascara_inundacion = (
        agua_despues
        .And(cambio_fuerte)
        .And(mascara_pendiente)
        .And(agua_permanente.Not())
        .rename("mascara_inundacion")
        .selfMask()
    )

    # -----------------------------------------------------
    # Interseccion con mascara agricola
    # -----------------------------------------------------
    mascara_agricola = construir_mascara_agricola(aoi)

    mascara_inundacion_agricola = (
        mascara_inundacion
        .And(mascara_agricola.eq(1))
        .rename("mascara_inundacion_agricola")
        .selfMask()
    )

    # -----------------------------------------------------
    # Area para filtrar ruido
    # -----------------------------------------------------
    area_m2 = ee.Number(
        mascara_inundacion_agricola
        .multiply(ee.Image.pixelArea())
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=aoi,
            scale=ESCALA_EXPORTACION,
            maxPixels=1e13,
            bestEffort=True
        )
        .get("mascara_inundacion_agricola")
    ).getInfo()

    if area_m2 is None:
        return {
            "estado": "mascara_vacia",
            "escenas_antes": n_antes,
            "escenas_despues": n_despues
        }

    area_ha = area_m2 / 10000.0

    if area_ha < AREA_MINIMA_HA:
        return {
            "estado": "mascara_muy_pequena",
            "escenas_antes": n_antes,
            "escenas_despues": n_despues,
            "area_ha": area_ha
        }

    # -----------------------------------------------------
    # Bounding box util de la propia mascara
    # -----------------------------------------------------
    vectores = mascara_inundacion_agricola.reduceToVectors(
        geometry=aoi,
        scale=ESCALA_EXPORTACION,
        geometryType="polygon",
        eightConnected=True,
        reducer=ee.Reducer.countEvery(),
        maxPixels=1e13,
        bestEffort=True
    )

    n_vectores = vectores.size().getInfo()

    if n_vectores == 0:
        return {
            "estado": "sin_vectores",
            "escenas_antes": n_antes,
            "escenas_despues": n_despues,
            "area_ha": area_ha
        }

    region_util = vectores.geometry().bounds(BUFFER_BBOX_UTIL_METROS)

    return {
        "estado": "ok",
        "imagen": mascara_inundacion_agricola.toByte(),
        "region": region_util,
        "escenas_antes": n_antes,
        "escenas_despues": n_despues,
        "area_ha": area_ha
    }


def exportar_imagen(imagen, nombre_salida, region):
    task = ee.batch.Export.image.toDrive(
        image=imagen,
        description=nombre_salida,
        folder=CARPETA_DRIVE,
        fileNamePrefix=nombre_salida,
        region=region,
        scale=ESCALA_EXPORTACION,
        crs="EPSG:4326",
        maxPixels=1e13,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True},
    )
    task.start()
    return task

# =========================================================
# 6) PROCESAMIENTO DEL LOTE
# =========================================================
resumen = []
tareas = []

for _, fila in eventos.iterrows():
    nombre_evento = fila["nombre_evento"]
    codigo_municipio = str(fila["codigo_municipio"]).zfill(5)
    fecha_evento = fila["fecha_evento"]

    print(f"\nProcesando: {nombre_evento}")

    resultado = construir_mascara_inundacion_agricola(
        codigo_municipio=codigo_municipio,
        fecha_evento=fecha_evento
    )

    estado = resultado["estado"]

    if estado != "ok":
        print("  ->", estado)
        resumen.append({
            "nombre_evento": nombre_evento,
            "codigo_municipio": codigo_municipio,
            "fecha_evento": str(fecha_evento)[:10],
            "estado": estado,
            "escenas_antes": resultado.get("escenas_antes"),
            "escenas_despues": resultado.get("escenas_despues"),
            "area_ha": resultado.get("area_ha")
        })
        continue

    tarea = exportar_imagen(
        resultado["imagen"],
        f"{nombre_evento}_mascara_inundacion_agricola",
        resultado["region"]
    )
    tareas.append(tarea)

    print(
        f"  -> exportado | escenas antes: {resultado['escenas_antes']} | "
        f"escenas despues: {resultado['escenas_despues']} | "
        f"area_ha: {resultado['area_ha']:.2f}"
    )

    resumen.append({
        "nombre_evento": nombre_evento,
        "codigo_municipio": codigo_municipio,
        "fecha_evento": str(fecha_evento)[:10],
        "estado": "exportado",
        "escenas_antes": resultado["escenas_antes"],
        "escenas_despues": resultado["escenas_despues"],
        "area_ha": resultado["area_ha"]
    })

    # pausa corta para no disparar todo de golpe
    time.sleep(1)

# Guardar resumen local
df_resumen = pd.DataFrame(resumen)
nombre_resumen = f"resumen_exportacion_mascara_agricola_{LOTE_INICIO}_{LOTE_FIN}.csv"
df_resumen.to_csv(nombre_resumen, index=False, encoding="utf-8-sig")

print("\nTareas lanzadas:", len(tareas))
print("Resumen guardado en:", nombre_resumen)
print(df_resumen.head())

Eventos en lote: 0
Empty DataFrame
Columns: [fecha_evento, codigo_municipio, DEPARTAMENTO, municipio, EVENTO, CODIGO EMERGENCIA, PERSONAS, FAMILIAS, HECTAREAS, VIV.DESTRU., VIV.AVER., VIAS, ACUED., OTROS, COMENTARIOS, VALOR.1, year, month, ym, missing_divipola_flag, flood_crop_text_flag, hectareas_reported_flag, flood_crop_quantified_flag, flood_crop_any_flag, event_id_clean, housing_damage_proxy, infra_damage_proxy, support_fngrd_value, nombre_evento]
Index: []

[0 rows x 29 columns]

Tareas lanzadas: 0
Resumen guardado en: resumen_exportacion_mascara_agricola_900_950.csv
Empty DataFrame
Columns: []
Index: []


# Descarga de datos/registros

In [ ]:
df['year'].unique()

array([2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014])

In [ ]:
dsfsd# Llave de control
nombre = 'datos_inundaciones_v2.xlsx'
descargar_xlsx(df, nombre)

NameError: name 'dsfsd' is not defined